# Mapa semántico de todas las tesis del CIDE

Este cuaderno construye un mapa semántico conjunto de las tesis de licenciatura, maestría y doctorado del CIDE. Combina embeddings multilingües, UMAP, clustering, topic modeling, evolución temporal, composición por programa y nivel, interdisciplinariedad y redes de profesorado homologado.

La entrada canónica es `tesis_cide.parquet`. No hace scraping: para actualizar datos, corre primero `make scrape`.


## Dependencias y configuración

El modelo por defecto es `sentence-transformers/paraphrase-multilingual-mpnet-base-v2`: es más pesado que MiniLM, pero en el benchmark local separó mejor los temas y mezcló mejor español/inglés. Puedes cambiar el modelo con `ST_MODEL_NAME`, el número de clusters con `ST_CLUSTER_K` y el batch size con `ST_BATCH_SIZE` sin editar la notebook.


In [ ]:
import os
import re
import json
import html
import hashlib
import warnings
import unicodedata
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
from collections import Counter

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import umap
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from IPython.display import display, HTML
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.manifold import trustworthiness
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

RANDOM_STATE = 420
DATA_PATH = Path('tesis_cide.parquet')
EMBEDDINGS_PATH = Path('embeddings_tesis.parquet')
CLUSTERS_PATH = Path('clusters_tesis.parquet')
CLUSTER_SUMMARY_PATH = Path('clusters_resumen.parquet')
CLUSTER_DIAGNOSTICS_PATH = Path('cluster_diagnostics.parquet')
UMAP_DIAGNOSTICS_PATH = Path('umap_diagnostics.parquet')
CLUSTER_VARIANTS_PATH = Path('cluster_variants.parquet')
CLUSTER_VARIANT_SUMMARY_PATH = Path('cluster_variant_summary.parquet')
CLUSTER_VARIANT_METRICS_PATH = Path('cluster_variant_metrics.parquet')
CLUSTER_KEYWORD_OVERLAP_PATH = Path('cluster_keyword_overlap.parquet')
TOPIC_MODEL_METRICS_PATH = Path('topic_model_metrics.parquet')
TOPIC_MODEL_FRONTIER_PATH = Path('topic_model_frontier.parquet')
TOPIC_STABILITY_PATH = Path('topic_stability.parquet')
TOPIC_MEMBERSHIP_PATH = Path('topic_memberships.parquet')
TOPIC_TAXONOMY_PATH = Path('topic_taxonomy.parquet')
TOPIC_KEYWORD_ALIASES_PATH = Path('data-raw/topic_keyword_aliases.csv')
CLUSTER_YEAR_PATH = Path('cluster_anio.parquet')
CLUSTER_LANGUAGE_PATH = Path('cluster_idioma.parquet')
CLUSTER_PROGRAM_PATH = Path('cluster_programa.parquet')
CLUSTER_LEVEL_PATH = Path('cluster_nivel.parquet')
CLUSTER_INTERDISCIPLINARITY_PATH = Path('cluster_interdisciplinariedad.parquet')
PROGRAM_CLUSTER_SUMMARY_PATH = Path('programa_cluster_resumen.parquet')
PROGRAM_SIMILARITY_PATH = Path('programa_similitud.parquet')
PROGRAM_YEAR_PATH = Path('programa_anio.parquet')
CLUSTER_LIFECYCLE_PATH = Path('cluster_lifecycle.parquet')
ADVISOR_CLUSTER_PATH = Path('asesor_cluster_resumen.parquet')
ADVISOR_SUMMARY_PATH = Path('asesor_resumen.parquet')
MODEL_BENCHMARK_PATH = Path('model_benchmark.parquet')
DASHBOARD_PATH = Path('semantic_dashboard.html')

MODEL_NAME = os.environ.get('ST_MODEL_NAME', 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
CLUSTER_K = int(os.environ.get('ST_CLUSTER_K', '20'))
TOPIC_MODEL_K = int(os.environ.get('ST_TOPIC_MODEL_K', '32'))
STABILITY_BOOTSTRAPS = int(os.environ.get('ST_TOPIC_STABILITY_BOOTSTRAPS', '6'))
DEVICE = os.environ.get('ST_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = int(os.environ.get('ST_BATCH_SIZE', '8' if 'mpnet' in MODEL_NAME.lower() else '16'))

pio.templates.default = 'plotly_white'
PALETTE = [
    '#2563eb', '#dc2626', '#059669', '#7c3aed', '#d97706', '#0891b2', '#be123c', '#4d7c0f',
    '#9333ea', '#0f766e', '#b45309', '#475569', '#db2777', '#64748b', '#ea580c', '#16a34a',
    '#1d4ed8', '#a16207', '#0369a1', '#c026d3', '#52525b', '#15803d'
]
LANGUAGE_COLOR_MAP = {'spa': '#2563eb', 'eng': '#f59e0b', 'desconocido': '#94a3b8'}
LEVEL_COLOR_MAP = {'Licenciatura': '#2563eb', 'Maestría': '#d97706', 'Doctorado': '#059669'}

print(f'Modelo de embeddings: {MODEL_NAME}')
print(f'Dispositivo: {DEVICE}; batch_size={BATCH_SIZE}')
print(f'Clusters configurados: k={CLUSTER_K}')


## Cargar y preparar los datos

Se combinan título y resumen. La columna `asesor_unificado` ya viene homologada desde el pipeline de scraping.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'No encontré {DATA_PATH}. Corre primero ScrapingTesisLicEcoCIDE.qmd o make scrape para generar el Parquet canónico.'
    )

df_raw = pd.read_parquet(DATA_PATH)

required_cols = {
    'anio_pub', 'autorx', 'titulo', 'resumen', 'materias', 'asesor_unificado',
    'idioma', 'nivel', 'programa', 'grado_programa', 'url'
}
missing_cols = required_cols.difference(df_raw.columns)
if missing_cols:
    raise ValueError(f'Faltan columnas requeridas en {DATA_PATH}: {sorted(missing_cols)}')

df_raw['anio_pub'] = pd.to_numeric(df_raw['anio_pub'], errors='coerce').astype('Int64')
df = df_raw.dropna(subset=['titulo']).copy()
for col in ['resumen', 'materias']:
    df[col] = df[col].fillna('')

df['texto'] = (
    df['titulo'].fillna('') + '. ' +
    df['resumen'].fillna('') + '. ' +
    df['materias'].fillna('')
).str.replace(r'\s+', ' ', regex=True).str.strip()
df = df[df['texto'].str.len() > 20].reset_index(drop=True)

df['asesor_unificado'] = df['asesor_unificado'].fillna('Asesor desconocido').replace('', 'Asesor desconocido')
df['idioma'] = df['idioma'].fillna('desconocido').replace('', 'desconocido')
df['nivel'] = df['nivel'].fillna('Desconocido').replace('', 'Desconocido')
df['programa'] = df['programa'].fillna('Programa desconocido').replace('', 'Programa desconocido')
df['grado_programa'] = df['grado_programa'].fillna(df['nivel'] + ' en ' + df['programa'])

df['periodo'] = pd.cut(
    df['anio_pub'].astype('float'),
    bins=[1969, 1979, 1989, 1999, 2009, 2019, 2029, 2039],
    labels=['1970s', '1980s', '1990s', '2000s', '2010s', '2020s', '2030s'],
)
df['text_hash'] = df['texto'].map(lambda x: hashlib.sha256(x.encode('utf-8')).hexdigest())

print(f'Tesis disponibles para analizar: {len(df):,}')
print(f'Años: {df["anio_pub"].min()}-{df["anio_pub"].max()}')
print(f'Niveles: {df["nivel"].value_counts().to_dict()}')
print(f'Programas por nivel: {df["grado_programa"].nunique()}')
n_advisors = df['asesor_unificado'].str.split(r'\s*\|\s*').explode().nunique()
print(f'Profesorado homologado distinto: {n_advisors}')
print(f'Idiomas detectados: {df["idioma"].value_counts().to_dict()}')
display(df[['anio_pub', 'nivel', 'programa', 'titulo', 'asesor_unificado', 'idioma', 'url']].head(5))


## Benchmark de modelos

El benchmark compara modelos multilingües sin volver a scrapear. La selección por defecto privilegia más estructura semántica y menos dependencia del idioma, aunque deja `ST_CLUSTER_K` como control explícito.


In [ ]:
if MODEL_BENCHMARK_PATH.exists():
    model_benchmark = pd.read_parquet(MODEL_BENCHMARK_PATH)
    benchmark_is_current = (
        'n_documents' in model_benchmark.columns
        and model_benchmark['n_documents'].eq(len(df)).any()
    )
    if benchmark_is_current:
        model_benchmark = model_benchmark[model_benchmark['n_documents'].eq(len(df))].copy()
        benchmark_cols = [
            'model_name', 'n_documents', 'embedding_dim', 'seconds', 'k', 'silhouette_cosine',
            'davies_bouldin', 'calinski_harabasz', 'language_nmi', 'min_cluster_size',
            'max_cluster_size', 'umap_trustworthiness'
        ]
        benchmark_cols = [c for c in benchmark_cols if c in model_benchmark.columns]
        benchmark_view = model_benchmark[benchmark_cols].sort_values(
            ['silhouette_cosine', 'umap_trustworthiness'], ascending=[False, False]
        ).reset_index(drop=True)
        display(benchmark_view.head(15))
    else:
        benchmark_view = pd.DataFrame()
        print('El benchmark existente corresponde a otra muestra; se omite del ranking global.')
else:
    model_benchmark = pd.DataFrame()
    benchmark_view = pd.DataFrame()
    print(f'No encontré {MODEL_BENCHMARK_PATH}.')


## Generar o cargar embeddings multilingües

Los embeddings se guardan en Parquet para no recalcularlos si el texto y el modelo no cambiaron. Esto permite iterar sobre UMAP, clustering y visualizaciones sin volver a descargar ni ejecutar el transformer.


In [ ]:
def embedding_columns(frame):
    return [c for c in frame.columns if c.startswith('embedding_')]

cache_ok = False
if EMBEDDINGS_PATH.exists():
    cached = pd.read_parquet(EMBEDDINGS_PATH)
    emb_cols = embedding_columns(cached)
    cache_ok = (
        len(cached) == len(df)
        and emb_cols
        and set(['url', 'text_hash', 'model_name']).issubset(cached.columns)
        and cached['url'].tolist() == df['url'].tolist()
        and cached['text_hash'].tolist() == df['text_hash'].tolist()
        and cached['model_name'].nunique() == 1
        and cached['model_name'].iloc[0] == MODEL_NAME
    )

if cache_ok:
    embeddings = cached[emb_cols].to_numpy(dtype=np.float32)
    print(f'Embeddings cargados desde cache: {EMBEDDINGS_PATH} {embeddings.shape}')
else:
    model = SentenceTransformer(MODEL_NAME, device=DEVICE)
    encode_texts = df['texto'].tolist()
    if 'multilingual-e5' in MODEL_NAME.lower() or MODEL_NAME.lower().startswith('intfloat/e5'):
        encode_texts = [f'passage: {text}' for text in encode_texts]

    embeddings = model.encode(
        encode_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    emb_cols = [f'embedding_{i:03d}' for i in range(embeddings.shape[1])]
    embeddings_export = pd.DataFrame(embeddings, columns=emb_cols)
    embeddings_export.insert(0, 'model_name', MODEL_NAME)
    embeddings_export.insert(0, 'text_hash', df['text_hash'])
    embeddings_export.insert(0, 'url', df['url'])
    embeddings_export.to_parquet(EMBEDDINGS_PATH, index=False)
    print(f'Embeddings generados y guardados: {EMBEDDINGS_PATH.resolve()} {embeddings.shape}')

if not np.all(np.isfinite(embeddings)):
    raise ValueError('Los embeddings contienen valores no finitos.')


## Reducir dimensionalidad con UMAP

UMAP se usa como layout visual. El clustering principal se calcula sobre embeddings originales para evitar que la geometría 2D determine artificialmente los grupos.


In [ ]:
umap_2d_model = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.04,
    metric='cosine',
    random_state=RANDOM_STATE,
)
umap_2d = umap_2d_model.fit_transform(embeddings)
umap_trust_2d = trustworthiness(embeddings, umap_2d, n_neighbors=30, metric='cosine')

umap_3d_model = umap.UMAP(
    n_components=3,
    n_neighbors=30,
    min_dist=0.04,
    metric='cosine',
    random_state=RANDOM_STATE,
)
umap_3d = umap_3d_model.fit_transform(embeddings)
umap_trust_3d = trustworthiness(embeddings, umap_3d, n_neighbors=30, metric='cosine')

# Backward-compatible name for existing downstream cells that refer to the main 2D visual layout.
umap_trust = umap_trust_2d

umap_diagnostics = pd.DataFrame([
    {
        'projection': 'umap_2d',
        'n_components': 2,
        'trustworthiness': umap_trust_2d,
        'n_neighbors': 30,
        'min_dist': 0.04,
        'metric': 'cosine',
        'interpretation': 'Mapa principal: más fácil de leer y comparar en tablero.',
    },
    {
        'projection': 'umap_3d',
        'n_components': 3,
        'trustworthiness': umap_trust_3d,
        'n_neighbors': 30,
        'min_dist': 0.04,
        'metric': 'cosine',
        'interpretation': 'Vista exploratoria: suele preservar mejor vecindades, pero requiere rotación interactiva.',
    },
])
umap_diagnostics['trustworthiness_delta_vs_2d'] = umap_diagnostics['trustworthiness'] - umap_trust_2d
umap_diagnostics['generated_at'] = datetime.now(timezone.utc).isoformat()
umap_diagnostics.to_parquet(UMAP_DIAGNOSTICS_PATH, index=False)

fig_umap_quality = px.bar(
    umap_diagnostics,
    x='projection',
    y='trustworthiness',
    text='trustworthiness',
    color='projection',
    color_discrete_map={'umap_2d': '#2563eb', 'umap_3d': '#d97706'},
    title='Calidad local de UMAP: 2D vs 3D',
    labels={'projection': 'Proyección', 'trustworthiness': 'Trustworthiness'},
    width=760,
    height=420,
)
fig_umap_quality.update_traces(texttemplate='%{text:.3f}', textposition='outside', marker_line_width=0)
fig_umap_quality.update_layout(showlegend=False, yaxis_range=[0, 1], margin=dict(l=55, r=25, t=70, b=45))
fig_umap_quality.show()

df_layout = df.copy()
df_layout['umap_x'] = umap_2d[:, 0]
df_layout['umap_y'] = umap_2d[:, 1]
df_layout['umap_z'] = umap_3d[:, 2]
print(f'Trustworthiness UMAP 2D: {umap_trust_2d:.4f}')
print(f'Trustworthiness UMAP 3D: {umap_trust_3d:.4f} (delta {umap_trust_3d - umap_trust_2d:+.4f})')
display(df_layout[['titulo', 'anio_pub', 'idioma', 'umap_x', 'umap_y', 'umap_z']].head(3))
display(umap_diagnostics)


## Diagnóstico de número de clusters

Se comparan K-Means sobre embeddings originales y sobre UMAP 2D para varios valores de `k`. La selección automática usa silhouette sobre embeddings, que es más cercana al espacio semántico real.


In [ ]:
language_labels = df_layout['idioma'].fillna('desconocido').astype(str)
program_labels = df_layout['grado_programa'].fillna('programa desconocido').astype(str)
level_labels = df_layout['nivel'].fillna('nivel desconocido').astype(str)

def evaluate_kmeans(features, feature_space, k_values, metric_for_silhouette, projection_trustworthiness=np.nan):
    rows = []
    for k in k_values:
        if k >= len(features):
            continue
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
        labels = km.fit_predict(features)
        silhouette_value = silhouette_score(features, labels, metric=metric_for_silhouette)
        rows.append({
            'model_name': MODEL_NAME,
            'feature_space': feature_space,
            'k': k,
            'silhouette': silhouette_value,
            'silhouette_metric': metric_for_silhouette,
            'davies_bouldin': davies_bouldin_score(features, labels),
            'calinski_harabasz': calinski_harabasz_score(features, labels),
            'inertia': km.inertia_,
            'language_nmi': normalized_mutual_info_score(language_labels, labels) if language_labels.nunique() > 1 else np.nan,
            'program_nmi': normalized_mutual_info_score(program_labels, labels) if program_labels.nunique() > 1 else np.nan,
            'level_nmi': normalized_mutual_info_score(level_labels, labels) if level_labels.nunique() > 1 else np.nan,
            'min_cluster_size': int(pd.Series(labels).value_counts().min()),
            'max_cluster_size': int(pd.Series(labels).value_counts().max()),
            'umap_trustworthiness': projection_trustworthiness,
        })
    return pd.DataFrame(rows)

k_values = [k for k in range(8, min(31, len(df_layout)), 2)]
diagnostics = pd.concat(
    [
        evaluate_kmeans(embeddings, 'embeddings', k_values, 'cosine'),
        evaluate_kmeans(umap_2d, 'umap_2d', k_values, 'euclidean', projection_trustworthiness=umap_trust_2d),
        evaluate_kmeans(umap_3d, 'umap_3d', k_values, 'euclidean', projection_trustworthiness=umap_trust_3d),
    ],
    ignore_index=True,
)

embedding_diag = diagnostics[diagnostics['feature_space'] == 'embeddings'].copy()
best_row = embedding_diag.sort_values(['silhouette', 'davies_bouldin'], ascending=[False, True]).iloc[0]
best_silhouette_k = int(best_row['k'])
selected_k = int(min(max(CLUSTER_K, 2), len(df_layout) - 1))

diagnostics['selected'] = diagnostics['feature_space'].eq('embeddings') & diagnostics['k'].eq(selected_k)
diagnostics['best_silhouette'] = diagnostics['feature_space'].eq('embeddings') & diagnostics['k'].eq(best_silhouette_k)
diagnostics['selection_reason'] = diagnostics['k'].map(lambda k: 'k configurado por ST_CLUSTER_K' if k == selected_k else '')
diagnostics['generated_at'] = datetime.now(timezone.utc).isoformat()
diagnostics.to_parquet(CLUSTER_DIAGNOSTICS_PATH, index=False)

print(f'k seleccionado: {selected_k} (configurado; KMeans sobre embeddings)')
print(f'k con mejor silhouette en embeddings: {best_silhouette_k}')
display(diagnostics.sort_values(['feature_space', 'k']))

fig_diag = px.line(
    diagnostics,
    x='k',
    y='silhouette',
    color='feature_space',
    markers=True,
    title='Diagnóstico de clusters por silhouette',
    width=980,
    height=450,
    labels={'silhouette': 'Silhouette', 'feature_space': 'Espacio'},
)
fig_diag.add_vline(x=selected_k, line_dash='dash', line_color='#111827', annotation_text=f'k usado: {selected_k}')
fig_diag.add_vline(x=best_silhouette_k, line_dash='dot', line_color='#dc2626', annotation_text=f'mejor silhouette: {best_silhouette_k}')
fig_diag.update_layout(legend_title_text='Espacio', margin=dict(l=40, r=30, t=70, b=40))
fig_diag.show()


## Clustering semántico sobre embeddings

El cluster final se calcula en el espacio de embeddings multilingües. UMAP queda como visualización.


In [ ]:
cluster_solutions = {
    'embeddings': {
        'variant_label': 'KMeans sobre embeddings',
        'feature_space': 'embeddings',
        'features': embeddings,
        'description': 'Cluster principal: usa el espacio semántico original.',
    },
    'umap_2d': {
        'variant_label': 'KMeans sobre UMAP 2D',
        'feature_space': 'umap_2d',
        'features': umap_2d,
        'description': 'Experimento: agrupa sobre el mapa 2D visible.',
    },
    'umap_3d': {
        'variant_label': 'KMeans sobre UMAP 3D',
        'feature_space': 'umap_3d',
        'features': umap_3d,
        'description': 'Experimento: agrupa sobre la proyección 3D.',
    },
}

cluster_variant_models = {}
cluster_variant_labels = {}
for variant, spec in cluster_solutions.items():
    model = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=60)
    labels = model.fit_predict(spec['features'])
    cluster_variant_models[variant] = model
    cluster_variant_labels[variant] = labels
    df_layout[f'cluster_id_{variant}'] = labels
    df_layout[f'cluster_label_{variant}'] = pd.Series(labels, index=df_layout.index).map(lambda x: f'Cluster {x:02d}')


embedding_topic_clusterers = {
    'spherical_kmeans_embeddings': {
        'variant_label': f'Spherical KMeans embeddings k={TOPIC_MODEL_K}',
        'feature_space': 'normalized_embeddings',
        'features': normalize(embeddings),
        'description': 'Text clustering clásico: K-Means sobre embeddings normalizados en la esfera coseno.',
        'algorithm': 'spherical_kmeans',
        'n_clusters': TOPIC_MODEL_K,
    },
}
for variant, spec in embedding_topic_clusterers.items():
    model = KMeans(n_clusters=spec['n_clusters'], random_state=RANDOM_STATE, n_init=60)
    labels = model.fit_predict(spec['features'])
    cluster_solutions[variant] = spec
    cluster_variant_models[variant] = model
    cluster_variant_labels[variant] = labels
    df_layout[f'cluster_id_{variant}'] = labels
    df_layout[f'cluster_label_{variant}'] = pd.Series(labels, index=df_layout.index).map(lambda x: f'Cluster {x:02d}')

agglomerative_features = normalize(embeddings)
agglomerative_model = AgglomerativeClustering(n_clusters=TOPIC_MODEL_K, metric='euclidean', linkage='ward')
agglomerative_labels = agglomerative_model.fit_predict(agglomerative_features)
cluster_solutions['agglomerative_embeddings'] = {
    'variant_label': f'Agglomerative Ward embeddings k={TOPIC_MODEL_K}',
    'feature_space': 'agglomerative_ward_normalized_embeddings',
    'features': agglomerative_features,
    'description': 'Clustering jerárquico aglomerativo Ward sobre embeddings normalizados.',
    'algorithm': 'agglomerative_ward',
    'n_clusters': TOPIC_MODEL_K,
}
cluster_variant_models['agglomerative_embeddings'] = agglomerative_model
cluster_variant_labels['agglomerative_embeddings'] = agglomerative_labels
df_layout['cluster_id_agglomerative_embeddings'] = agglomerative_labels
df_layout['cluster_label_agglomerative_embeddings'] = pd.Series(agglomerative_labels, index=df_layout.index).map(lambda x: f'Cluster {x:02d}')

cluster_labels = cluster_variant_labels['embeddings']
cluster_algo = 'kmeans (embeddings)'

df_layout['cluster_id'] = cluster_labels
df_layout['cluster_label'] = df_layout['cluster_id'].map(lambda x: f'Cluster {x:02d}')
cluster_order = [f'Cluster {cid:02d}' for cid in sorted(df_layout['cluster_id'].unique())]
cluster_color_map = {label: PALETTE[i % len(PALETTE)] for i, label in enumerate(cluster_order)}

print(f'Algoritmo de clustering activo: {cluster_algo} (k={selected_k})')
print('Variantes experimentales calculadas:')
for variant, spec in cluster_solutions.items():
    sizes = pd.Series(cluster_variant_labels[variant]).value_counts().sort_index()
    print(f"- {spec['variant_label']}: {len(sizes)} clusters; tamaño min={int(sizes.min())}, max={int(sizes.max())}")

cluster_summary_base = (
    df_layout.groupby(['cluster_label', 'cluster_id'])
    .agg(
        conteo=('titulo', 'count'),
        anio_promedio=('anio_pub', 'mean'),
        anio_min=('anio_pub', 'min'),
        anio_max=('anio_pub', 'max'),
        idiomas=('idioma', lambda s: ', '.join(s.value_counts().head(3).index)),
        niveles=('nivel', lambda s: ', '.join(s.value_counts().index)),
        programas_top=('grado_programa', lambda s: ' | '.join(s.value_counts().head(4).index)),
        n_programas=('grado_programa', 'nunique'),
    )
    .reset_index()
    .sort_values('conteo', ascending=False)
)

display(cluster_summary_base)


## Etiquetar clusters con keywords y tesis representativas

Se usa TF-IDF por cluster para términos característicos y similitud al centroide para tesis representativas.


In [ ]:
SPANISH_STOPWORDS = {
    'a', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'como', 'con', 'contra', 'cual', 'cuando',
    'de', 'del', 'desde', 'donde', 'dos', 'durante', 'e', 'el', 'ella', 'ellas', 'ellos', 'en', 'entre',
    'era', 'eran', 'es', 'esa', 'esas', 'ese', 'eso', 'esos', 'esta', 'estaba', 'estado', 'estados',
    'estan', 'estar', 'estas', 'este', 'esto', 'estos', 'fue', 'fueron', 'ha', 'hacia', 'han', 'hasta',
    'hay', 'la', 'las', 'le', 'les', 'lo', 'los', 'mas', 'más', 'me', 'mediante', 'mi', 'mientras',
    'muy', 'no', 'nos', 'o', 'para', 'pero', 'por', 'porque', 'que', 'se', 'sin', 'sobre', 'son',
    'su', 'sus', 'tambien', 'también', 'te', 'tiene', 'tienen', 'un', 'una', 'uno', 'unos', 'y', 'ya',
    'méxico', 'mexico', 'cide', 'tesis', 'licenciatura', 'economía', 'economia', 'análisis', 'analisis',
    'modelo', 'modelos', 'caso', 'estudio', 'evidencia', 'efecto', 'efectos', 'datos', 'resultados',
    'trabajo', 'seccion', 'sección', 'capitulo', 'capítulo', 'presente', 'objetivo', 'objetivos',
    'literatura', 'investigacion', 'investigación', 'variables', 'variable', 'muestra', 'metodologia',
    'metodología', 'using', 'based', 'paper', 'thesis', 'chapter', 'section'
}
STOPWORDS = sorted(set(ENGLISH_STOP_WORDS).union(SPANISH_STOPWORDS))

def normalize_keyword_key(value):
    text = unicodedata.normalize('NFKD', str(value).lower())
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^a-z0-9\s]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

if TOPIC_KEYWORD_ALIASES_PATH.exists():
    topic_keyword_aliases = pd.read_csv(TOPIC_KEYWORD_ALIASES_PATH)
else:
    topic_keyword_aliases = pd.DataFrame(columns=['alias', 'canonical', 'domain'])

for col in ['alias', 'canonical']:
    if col not in topic_keyword_aliases.columns:
        topic_keyword_aliases[col] = ''

topic_keyword_aliases = topic_keyword_aliases.dropna(subset=['alias', 'canonical']).copy()
topic_keyword_aliases['alias_key'] = topic_keyword_aliases['alias'].map(normalize_keyword_key)
topic_keyword_aliases['canonical_clean'] = topic_keyword_aliases['canonical'].astype(str).str.strip().str.lower()
keyword_alias_map = {
    row.alias_key: row.canonical_clean
    for row in topic_keyword_aliases.itertuples(index=False)
    if row.alias_key and row.canonical_clean
}


def canonicalize_keyword(term):
    key = normalize_keyword_key(term)
    if not key:
        return ''
    if key in keyword_alias_map:
        return keyword_alias_map[key]
    tokenized = key.split()
    if len(tokenized) == 1:
        return tokenized[0]
    mapped_tokens = [keyword_alias_map.get(token, token) for token in tokenized]
    collapsed = ' '.join(mapped_tokens)
    return keyword_alias_map.get(normalize_keyword_key(collapsed), collapsed)


def dedupe_canonical_keywords(keyword_terms, n=12):
    selected = []
    seen = set()
    for term in keyword_terms:
        canonical = canonicalize_keyword(term)
        if canonical and canonical not in seen and canonical not in STOPWORDS:
            selected.append(canonical)
            seen.add(canonical)
        if len(selected) >= n:
            break
    return selected

print(f'Aliases bilingues de keywords cargados: {len(keyword_alias_map)}')


vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words=STOPWORDS,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.85,
    max_features=12000,
)

tfidf = vectorizer.fit_transform(df_layout['texto'])
terms = np.array(vectorizer.get_feature_names_out())
embeddings_norm = normalize(embeddings)

count_vectorizer = CountVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words=STOPWORDS,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.85,
)
count_matrix = count_vectorizer.fit_transform(df_layout['texto'])
count_terms = np.array(count_vectorizer.get_feature_names_out())

nmf_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words=STOPWORDS,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    max_features=9000,
)
nmf_tfidf = nmf_vectorizer.fit_transform(df_layout['texto'])
nmf_terms = np.array(nmf_vectorizer.get_feature_names_out())
nmf_model = NMF(
    n_components=TOPIC_MODEL_K,
    init='random',
    solver='mu',
    beta_loss='kullback-leibler',
    max_iter=900,
    random_state=RANDOM_STATE,
)
nmf_doc_topic = nmf_model.fit_transform(nmf_tfidf)
nmf_labels = nmf_doc_topic.argmax(axis=1).astype(int)
cluster_variant_labels['nmf_tfidf'] = nmf_labels
df_layout['cluster_id_nmf_tfidf'] = nmf_labels
df_layout['cluster_label_nmf_tfidf'] = pd.Series(nmf_labels, index=df_layout.index).map(lambda x: f'Topic {x:02d}')
cluster_solutions['nmf_tfidf'] = {
    'variant_label': f'NMF sobre TF-IDF k={TOPIC_MODEL_K}',
    'feature_space': 'tfidf_nmf',
    'features': nmf_doc_topic,
    'description': 'Topic model matricial: NMF con divergencia KL sobre TF-IDF uni/bigramas.',
    'algorithm': 'nmf_tfidf',
    'n_clusters': TOPIC_MODEL_K,
    'topic_components': nmf_model.components_,
    'topic_terms': nmf_terms,
}

lda_model = LatentDirichletAllocation(
    n_components=TOPIC_MODEL_K,
    max_iter=80,
    learning_method='batch',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
lda_doc_topic = lda_model.fit_transform(count_matrix)
lda_labels = lda_doc_topic.argmax(axis=1).astype(int)
cluster_variant_labels['lda_count'] = lda_labels
df_layout['cluster_id_lda_count'] = lda_labels
df_layout['cluster_label_lda_count'] = pd.Series(lda_labels, index=df_layout.index).map(lambda x: f'Topic {x:02d}')
cluster_solutions['lda_count'] = {
    'variant_label': f'LDA conteos k={TOPIC_MODEL_K}',
    'feature_space': 'count_lda',
    'features': lda_doc_topic,
    'description': 'Topic model probabilístico generativo LDA sobre conteos.',
    'algorithm': 'lda_count',
    'n_clusters': TOPIC_MODEL_K,
    'topic_components': lda_model.components_,
    'topic_terms': count_terms,
}

topic_vectorizer = CountVectorizer(
    lowercase=True,
    strip_accents='unicode',
    stop_words=STOPWORDS,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.85,
)
bertopic_umap_model = umap.UMAP(
    n_components=10,
    n_neighbors=30,
    min_dist=0.0,
    metric='cosine',
    random_state=RANDOM_STATE,
)
bertopic_hdbscan_model = HDBSCAN(
    min_cluster_size=25,
    min_samples=2,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
)
bertopic_ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
bertopic_model = BERTopic(
    embedding_model=None,
    umap_model=bertopic_umap_model,
    hdbscan_model=bertopic_hdbscan_model,
    vectorizer_model=topic_vectorizer,
    ctfidf_model=bertopic_ctfidf_model,
    top_n_words=12,
    calculate_probabilities=False,
    language='multilingual',
    verbose=False,
)

bertopic_labels, _ = bertopic_model.fit_transform(df_layout['texto'].tolist(), embeddings=embeddings)
bertopic_labels = np.asarray(bertopic_labels, dtype=int)
cluster_variant_labels['bertopic_hdbscan'] = bertopic_labels
df_layout['cluster_id_bertopic_hdbscan'] = bertopic_labels
df_layout['cluster_label_bertopic_hdbscan'] = pd.Series(bertopic_labels, index=df_layout.index).map(
    lambda x: 'Outliers' if int(x) == -1 else f'Topic {int(x):02d}'
)
cluster_solutions['bertopic_hdbscan'] = {
    'variant_label': 'BERTopic + HDBSCAN',
    'feature_space': 'umap_10d_hdbscan',
    'features': getattr(bertopic_model.umap_model, 'embedding_', embeddings),
    'description': 'Topic model: embeddings transformer, UMAP 10D, HDBSCAN y c-TF-IDF.',
    'algorithm': 'bertopic_hdbscan',
}


CONSENSUS_VARIANTS = [
    'embeddings',
    'umap_2d',
    'umap_3d',
    'spherical_kmeans_embeddings',
    'agglomerative_embeddings',
    'nmf_tfidf',
    'bertopic_hdbscan',
]
CONSENSUS_MODEL_WEIGHTS = {
    'embeddings': 0.85,
    'umap_2d': 0.90,
    'umap_3d': 0.90,
    'spherical_kmeans_embeddings': 1.05,
    'agglomerative_embeddings': 1.20,
    'nmf_tfidf': 1.15,
    'bertopic_hdbscan': 1.25,
}


def build_consensus_similarity(labels_by_variant, variants, weights):
    n = len(df_layout)
    same_weight = np.zeros((n, n), dtype=np.float32)
    observed_weight = np.zeros((n, n), dtype=np.float32)
    for variant in variants:
        labels = np.asarray(labels_by_variant[variant])
        valid = labels >= 0
        if valid.sum() < 2:
            continue
        weight = float(weights.get(variant, 1.0))
        valid_pair = np.outer(valid, valid)
        same_pair = (labels[:, None] == labels[None, :]) & valid_pair
        same_weight += same_pair.astype(np.float32) * weight
        observed_weight += valid_pair.astype(np.float32) * weight
    similarity = np.divide(same_weight, observed_weight, out=np.zeros_like(same_weight), where=observed_weight > 0)
    np.fill_diagonal(similarity, 1.0)
    return similarity

consensus_similarity = build_consensus_similarity(cluster_variant_labels, CONSENSUS_VARIANTS, CONSENSUS_MODEL_WEIGHTS)
consensus_distance = 1.0 - consensus_similarity
consensus_distance = (consensus_distance + consensus_distance.T) / 2
np.fill_diagonal(consensus_distance, 0.0)
consensus_model = AgglomerativeClustering(n_clusters=TOPIC_MODEL_K, metric='precomputed', linkage='average')
consensus_labels = consensus_model.fit_predict(consensus_distance)
cluster_variant_labels['consensus_ensemble'] = consensus_labels
df_layout['cluster_id_consensus_ensemble'] = consensus_labels
df_layout['cluster_label_consensus_ensemble'] = pd.Series(consensus_labels, index=df_layout.index).map(lambda x: f'Topic {int(x):02d}')
cluster_solutions['consensus_ensemble'] = {
    'variant_label': f'Consenso ensemble k={TOPIC_MODEL_K}',
    'feature_space': 'consensus_coassociation',
    'features': consensus_similarity,
    'description': 'Ensemble: matriz de co-asignacion ponderada entre KMeans, Ward, NMF y BERTopic; reclustering aglomerativo average.',
    'algorithm': 'consensus_ensemble',
    'n_clusters': TOPIC_MODEL_K,
    'source_variants': ', '.join(CONSENSUS_VARIANTS),
}
print(f'Consenso ensemble calculado con {len(CONSENSUS_VARIANTS)} variantes y k={TOPIC_MODEL_K}.')

topic_info = bertopic_model.get_topic_info()
n_bertopic_topics = int((pd.Series(bertopic_labels) >= 0).nunique())
n_bertopic_outliers = int((bertopic_labels == -1).sum())
print(f'BERTopic + HDBSCAN detectó {n_bertopic_topics} tópicos y {n_bertopic_outliers} outliers.')
display(topic_info.head(12))

CURATED_CLUSTER_THEMES = {
    0: 'Educación y desempeño escolar',
    1: 'Crédito y sistema financiero',
    2: 'Industria, productividad e innovación',
    3: 'Macroeconomía y política económica',
    4: 'Género, familia y mercado laboral',
    5: 'Desarrollo, pobreza y migración',
    6: 'Crimen, seguridad y ciudad',
    7: 'Organización industrial y teoría de juegos',
    8: 'Finanzas, riesgo y mercados',
    9: 'Agricultura, medio ambiente y clima',
    10: 'Salud, bienestar y protección social',
}

CURATED_ALL_CIDE_THEMES = {
    0: 'Empresas, innovación y competencia',
    1: 'Crimen, violencia y seguridad',
    2: 'Gobierno, federalismo y política fiscal',
    3: 'Políticas públicas e implementación',
    4: 'Macroeconomía, inflación y econometría',
    5: 'Derecho internacional y derechos humanos',
    6: 'Corrupción y rendición de cuentas',
    7: 'Ciudad, municipios e infraestructura',
    8: 'Justicia, tribunales y derecho constitucional',
    9: 'Género, violencia y participación',
    10: 'Modelos matemáticos, mercados y juegos',
    11: 'Elecciones, partidos y comportamiento político',
    12: 'Crédito, inclusión y sistema financiero',
    13: 'Salud, pandemia y protección social',
    14: 'Riesgo, banca y mercados financieros',
    15: 'Educación, escuelas y desempeño',
    16: 'Historia, cultura y sociedad',
    17: 'Mercado laboral, movilidad y pobreza',
    18: 'Medio ambiente, agricultura y agua',
    19: 'Energía, electricidad y regulación',
}

TRUST_BY_SPACE = {
    'embeddings': np.nan,
    'umap_2d': umap_trust_2d,
    'umap_3d': umap_trust_3d,
    'umap_10d_hdbscan': np.nan,
}
WORD_RE = re.compile(r'[a-záéíóúüñ]{3,}', flags=re.IGNORECASE)

def topic_tokenize(text):
    tokens = []
    for token in WORD_RE.findall(str(text).lower()):
        token = token.strip()
        if token and token not in STOPWORDS:
            tokens.append(token)
    return tokens

topic_tokenized_docs = [topic_tokenize(text) for text in df_layout['texto']]
topic_dictionary = Dictionary(topic_tokenized_docs)


def top_keywords_for_mask(mask, n=12):
    scores = np.asarray(tfidf[mask].mean(axis=0)).ravel()
    top_idx = scores.argsort()[::-1]
    selected = [terms[i] for i in top_idx if scores[i] > 0][:n]
    return selected


def bertopic_keywords(cid, fallback_mask, n=12):
    if int(cid) == -1:
        return top_keywords_for_mask(fallback_mask, n=n)
    topic_words = bertopic_model.get_topic(int(cid)) or []
    selected = [word for word, score in topic_words if word][:n]
    return selected if selected else top_keywords_for_mask(fallback_mask, n=n)


def component_keywords(variant, cid, fallback_mask, n=12):
    spec = cluster_solutions.get(variant, {})
    components = spec.get('topic_components')
    term_array = spec.get('topic_terms')
    if components is None or term_array is None or int(cid) < 0:
        return top_keywords_for_mask(fallback_mask, n=n)
    scores = np.asarray(components[int(cid)]).ravel()
    top_idx = scores.argsort()[::-1]
    selected = [str(term_array[i]) for i in top_idx if scores[i] > 0][:n]
    return selected if selected else top_keywords_for_mask(fallback_mask, n=n)


def keywords_for_variant(variant, cid, fallback_mask, n=12):
    if variant == 'bertopic_hdbscan':
        raw_terms = bertopic_keywords(cid, fallback_mask, n=n * 2)
    elif 'topic_components' in cluster_solutions.get(variant, {}):
        raw_terms = component_keywords(variant, cid, fallback_mask, n=n * 2)
    else:
        raw_terms = top_keywords_for_mask(fallback_mask, n=n * 2)
    return dedupe_canonical_keywords(raw_terms, n=n)


def auto_cluster_theme(keyword_terms):
    pieces = [piece.strip() for piece in keyword_terms if piece.strip()]
    return ' / '.join(pieces[:3]).title() if pieces else 'Tema sin keywords claras'


def label_for_cluster(cid, variant):
    cid = int(cid)
    if cid == -1:
        return 'Outliers'
    return f'Topic {cid:02d}' if cluster_solutions.get(variant, {}).get('algorithm') in {'bertopic_hdbscan', 'nmf_tfidf', 'lda_count'} else f'Cluster {cid:02d}'


def representative_titles(mask_idx, n=3):
    if len(mask_idx) == 0:
        return ''
    centroid = normalize(embeddings[mask_idx].mean(axis=0, keepdims=True))
    sims = cosine_similarity(embeddings_norm[mask_idx], centroid).ravel()
    top_local = mask_idx[np.argsort(sims)[::-1][:n]]
    return ' | '.join(df_layout.loc[top_local, 'titulo'].tolist())


def top_advisors_for_mask(mask, n=5):
    advisors = (
        df_layout.loc[mask, 'asesor_unificado']
        .fillna('Sin dato')
        .str.split(r'\s*\|\s*')
        .explode()
        .str.strip()
    )
    counts = advisors[advisors.ne('')].value_counts().head(n)
    return ', '.join(f'{advisor} ({count})' for advisor, count in counts.items())


def build_cluster_profile(variant, labels, curated_themes=None):
    spec = cluster_solutions[variant]
    labels = np.asarray(labels)
    rows = []
    keyword_terms_by_cluster = {}
    keywords_by_cluster_local = {}
    theme_by_cluster = {}
    total = len(labels)

    for cid in sorted(np.unique(labels)):
        mask = labels == cid
        mask_idx = np.flatnonzero(mask)
        keyword_terms = keywords_for_variant(variant, cid, mask, n=12)
        keyword_terms_by_cluster[int(cid)] = keyword_terms
        keywords_by_cluster_local[int(cid)] = ', '.join(keyword_terms)
        if int(cid) == -1:
            theme = 'Sin tema robusto / outliers'
        else:
            theme = curated_themes.get(int(cid), auto_cluster_theme(keyword_terms)) if curated_themes else auto_cluster_theme(keyword_terms)
        theme_by_cluster[int(cid)] = theme
        cluster_label = label_for_cluster(cid, variant)
        years = df_layout.loc[mask, 'anio_pub']
        languages = df_layout.loc[mask, 'idioma'].fillna('desconocido').astype(str)
        rows.append({
            'variant': variant,
            'variant_label': spec['variant_label'],
            'feature_space': spec['feature_space'],
            'cluster_id': int(cid),
            'cluster_label': cluster_label,
            'cluster_theme': theme,
            'cluster_display_label': f'{cluster_label}: {theme}',
            'conteo': int(mask.sum()),
            'participacion': float(mask.sum() / total),
            'anio_promedio': float(years.mean()),
            'anio_min': int(years.min()) if years.notna().any() else np.nan,
            'anio_max': int(years.max()) if years.notna().any() else np.nan,
            'idiomas': ', '.join(languages.value_counts().head(3).index),
            'niveles': ', '.join(df_layout.loc[mask, 'nivel'].value_counts().index),
            'n_programas': int(df_layout.loc[mask, 'grado_programa'].nunique()),
            'programas_top': ' | '.join(df_layout.loc[mask, 'grado_programa'].value_counts().head(4).index),
            'keywords': ', '.join(keyword_terms),
            'tesis_representativas': representative_titles(mask_idx),
            'asesores_top': top_advisors_for_mask(mask),
            'cluster_algo': spec.get('algorithm', f"kmeans ({spec['feature_space']})"),
            'embedding_model': MODEL_NAME,
            'umap_trustworthiness': TRUST_BY_SPACE.get(spec['feature_space'], np.nan),
        })

    profile = pd.DataFrame(rows).sort_values('conteo', ascending=False).reset_index(drop=True)
    return profile, keywords_by_cluster_local, keyword_terms_by_cluster, theme_by_cluster


cluster_profiles = {}
keywords_by_variant = {}
keyword_terms_by_variant = {}
cluster_themes_by_variant = {}
for variant, labels in cluster_variant_labels.items():
    use_all_cide_curated = (
        variant == 'embeddings'
        and selected_k == 20
        and df_layout['grado_programa'].nunique() > 1
    )
    use_legacy_curated = (
        variant == 'embeddings'
        and selected_k == 11
        and df_layout['grado_programa'].nunique() == 1
    )
    if use_all_cide_curated:
        curated = CURATED_ALL_CIDE_THEMES
    elif use_legacy_curated:
        curated = CURATED_CLUSTER_THEMES
    else:
        curated = None
    profile, keyword_strings, keyword_terms, theme_map = build_cluster_profile(variant, labels, curated_themes=curated)
    cluster_profiles[variant] = profile
    keywords_by_variant[variant] = keyword_strings
    keyword_terms_by_variant[variant] = keyword_terms
    cluster_themes_by_variant[variant] = theme_map

cluster_variant_summary = pd.concat(cluster_profiles.values(), ignore_index=True)

# Variables canónicas usadas por las visualizaciones existentes.
keywords_by_cluster = keywords_by_variant['embeddings']
cluster_theme_by_id = cluster_themes_by_variant['embeddings']
cluster_overview = cluster_profiles['embeddings'].sort_values('conteo', ascending=False).reset_index(drop=True)

for variant, theme_map in cluster_themes_by_variant.items():
    id_col = f'cluster_id_{variant}'
    label_col = f'cluster_label_{variant}'
    theme_col = f'cluster_theme_{variant}'
    display_col = f'cluster_display_label_{variant}'
    df_layout[theme_col] = df_layout[id_col].map(theme_map)
    df_layout[display_col] = df_layout.apply(lambda r: f"{r[label_col]}: {r[theme_col]}", axis=1)

df_layout['cluster_theme'] = df_layout['cluster_theme_embeddings']
df_layout['cluster_display_label'] = df_layout['cluster_display_label_embeddings']

cluster_theme_order = [cluster_theme_by_id[cid] for cid in sorted(df_layout['cluster_id'].unique())]
cluster_display_order = [f"Cluster {cid:02d}: {cluster_theme_by_id[cid]}" for cid in sorted(df_layout['cluster_id'].unique())]
cluster_theme_color_map = {
    cluster_theme_by_id[cid]: cluster_color_map[f'Cluster {cid:02d}']
    for cid in sorted(df_layout['cluster_id'].unique())
}
cluster_display_color_map = {
    f"Cluster {cid:02d}: {cluster_theme_by_id[cid]}": cluster_color_map[f'Cluster {cid:02d}']
    for cid in sorted(df_layout['cluster_id'].unique())
}
cluster_label_to_theme = {f'Cluster {cid:02d}': theme for cid, theme in cluster_theme_by_id.items()}
cluster_label_to_display = {label: f'{label}: {theme}' for label, theme in cluster_label_to_theme.items()}

variant_theme_orders = {}
variant_theme_color_maps = {}
for variant, theme_map in cluster_themes_by_variant.items():
    ids = [cid for cid in sorted(theme_map) if cid != -1]
    if -1 in theme_map:
        ids.append(-1)
    variant_theme_orders[variant] = [theme_map[cid] for cid in ids]
    variant_theme_color_maps[variant] = {
        theme_map[cid]: ('#94a3b8' if cid == -1 else PALETTE[i % len(PALETTE)])
        for i, cid in enumerate(ids)
    }




consensus_topic_ids = sorted([cid for cid in np.unique(consensus_labels) if int(cid) >= 0])
consensus_member_index = {int(cid): np.flatnonzero(consensus_labels == cid) for cid in consensus_topic_ids}

membership_rows = []
for doc_idx in range(len(df_layout)):
    scores = []
    for cid in consensus_topic_ids:
        members = consensus_member_index[int(cid)]
        score = float(consensus_similarity[doc_idx, members].mean()) if len(members) else 0.0
        scores.append((int(cid), score))
    total_score = sum(score for _, score in scores) or 1.0
    for rank, (cid, score) in enumerate(sorted(scores, key=lambda item: item[1], reverse=True)[:3], start=1):
        membership_rows.append({
            'tesis_row_id': int(doc_idx),
            'titulo': df_layout.loc[doc_idx, 'titulo'],
            'autorx': df_layout.loc[doc_idx, 'autorx'],
            'anio_pub': df_layout.loc[doc_idx, 'anio_pub'],
            'idioma': df_layout.loc[doc_idx, 'idioma'],
            'nivel': df_layout.loc[doc_idx, 'nivel'],
            'programa': df_layout.loc[doc_idx, 'programa'],
            'grado_programa': df_layout.loc[doc_idx, 'grado_programa'],
            'rank': int(rank),
            'consensus_cluster_id': int(cid),
            'consensus_cluster_label': f'Topic {int(cid):02d}',
            'consensus_cluster_theme': cluster_themes_by_variant['consensus_ensemble'].get(int(cid), f'Topic {int(cid):02d}'),
            'membership_score': score,
            'membership_share': score / total_score,
        })

topic_memberships = pd.DataFrame(membership_rows)
wide_memberships = topic_memberships.pivot(index='tesis_row_id', columns='rank', values=['consensus_cluster_id', 'consensus_cluster_theme', 'membership_score', 'membership_share'])
wide_memberships.columns = [f'{metric}_rank_{rank}' for metric, rank in wide_memberships.columns]
wide_memberships = wide_memberships.reset_index()
df_layout = df_layout.merge(wide_memberships, left_index=True, right_on='tesis_row_id', how='left').sort_values('tesis_row_id').reset_index(drop=True)
df_layout['consensus_membership_margin'] = df_layout['membership_score_rank_1'].fillna(0) - df_layout['membership_score_rank_2'].fillna(0)
df_layout['consensus_primary_theme'] = df_layout['consensus_cluster_theme_rank_1']
df_layout['consensus_secondary_theme'] = df_layout['consensus_cluster_theme_rank_2']

macro_labels = np.asarray(cluster_variant_labels['embeddings'])
taxonomy_rows = []
for cid in consensus_topic_ids:
    mask = consensus_labels == cid
    macro_counts = pd.Series(macro_labels[mask]).value_counts()
    dominant_macro_id = int(macro_counts.index[0])
    dominant_macro_share = float(macro_counts.iloc[0] / macro_counts.sum())
    macro_theme = cluster_themes_by_variant['embeddings'].get(dominant_macro_id, f'Cluster {dominant_macro_id:02d}')
    subtheme = cluster_themes_by_variant['consensus_ensemble'].get(int(cid), f'Topic {int(cid):02d}')
    taxonomy_rows.append({
        'macro_cluster_id': dominant_macro_id,
        'macro_cluster_label': f'Cluster {dominant_macro_id:02d}',
        'macro_theme': macro_theme,
        'subtopic_id': int(cid),
        'subtopic_label': f'Topic {int(cid):02d}',
        'subtopic_theme': subtheme,
        'taxonomy_label': f'{macro_theme} > {subtheme}',
        'conteo': int(mask.sum()),
        'macro_share': dominant_macro_share,
        'keywords': keywords_by_variant['consensus_ensemble'].get(int(cid), ''),
        'tesis_representativas': representative_titles(np.flatnonzero(mask)),
    })

topic_taxonomy = pd.DataFrame(taxonomy_rows).sort_values(['macro_cluster_id', 'conteo'], ascending=[True, False]).reset_index(drop=True)
subtopic_to_taxonomy = topic_taxonomy.set_index('subtopic_id')['taxonomy_label'].to_dict()
subtopic_to_macro_theme = topic_taxonomy.set_index('subtopic_id')['macro_theme'].to_dict()
subtopic_to_macro_id = topic_taxonomy.set_index('subtopic_id')['macro_cluster_id'].to_dict()
df_layout['macro_cluster_id'] = df_layout['cluster_id_consensus_ensemble'].map(subtopic_to_macro_id)
df_layout['macro_cluster_theme'] = df_layout['cluster_id_consensus_ensemble'].map(subtopic_to_macro_theme)
df_layout['taxonomy_label'] = df_layout['cluster_id_consensus_ensemble'].map(subtopic_to_taxonomy)
print(f'Taxonomia jerarquica construida: {topic_taxonomy["macro_cluster_id"].nunique()} macrotemas y {len(topic_taxonomy)} subtemas de consenso.')


def keyword_overlap_rows(profile, keyword_terms_by_cluster):
    rows = []
    topic_profile = profile[profile['cluster_id'].ge(0)].sort_values('cluster_id')
    for left, right in combinations(topic_profile.itertuples(index=False), 2):
        left_terms = set(keyword_terms_by_cluster[int(left.cluster_id)][:10])
        right_terms = set(keyword_terms_by_cluster[int(right.cluster_id)][:10])
        union = left_terms | right_terms
        shared = sorted(left_terms & right_terms)
        rows.append({
            'variant': left.variant,
            'variant_label': left.variant_label,
            'feature_space': left.feature_space,
            'cluster_a': left.cluster_label,
            'cluster_b': right.cluster_label,
            'cluster_a_theme': left.cluster_theme,
            'cluster_b_theme': right.cluster_theme,
            'n_shared_keywords': len(shared),
            'keyword_jaccard': len(shared) / len(union) if union else 0.0,
            'shared_keywords': ', '.join(shared),
        })
    return pd.DataFrame(rows)

keyword_overlap_pairs = pd.concat(
    [keyword_overlap_rows(cluster_profiles[variant], keyword_terms_by_variant[variant]) for variant in cluster_solutions],
    ignore_index=True,
)


def topic_terms_for_coherence(keyword_terms_by_cluster):
    topics = []
    for cid, terms_local in keyword_terms_by_cluster.items():
        if int(cid) == -1:
            continue
        words = []
        seen = set()
        for term in terms_local[:15]:
            for token in WORD_RE.findall(str(term).lower()):
                if token in topic_dictionary.token2id and token not in seen and token not in STOPWORDS:
                    words.append(token)
                    seen.add(token)
            if len(words) >= 10:
                break
        if len(words) >= 2:
            topics.append(words[:10])
    return topics


topic_corpus = [topic_dictionary.doc2bow(tokens) for tokens in topic_tokenized_docs]


def coherence_score(keyword_terms_by_cluster, coherence='c_v'):
    coherence_topics = topic_terms_for_coherence(keyword_terms_by_cluster)
    if len(coherence_topics) < 2:
        return np.nan
    kwargs = {
        'topics': coherence_topics,
        'dictionary': topic_dictionary,
        'coherence': coherence,
    }
    if coherence == 'u_mass':
        kwargs['corpus'] = topic_corpus
    else:
        kwargs['texts'] = topic_tokenized_docs
    return float(CoherenceModel(**kwargs).get_coherence())


def coherence_cv(keyword_terms_by_cluster):
    return coherence_score(keyword_terms_by_cluster, coherence='c_v')


def embedding_cluster_scores(labels):
    labels = np.asarray(labels)
    valid = labels >= 0
    valid_labels = labels[valid]
    if valid.sum() <= 2 or len(np.unique(valid_labels)) < 2:
        return np.nan, np.nan, np.nan
    return (
        float(silhouette_score(embeddings[valid], valid_labels, metric='cosine')),
        float(davies_bouldin_score(embeddings_norm[valid], valid_labels)),
        float(calinski_harabasz_score(embeddings_norm[valid], valid_labels)),
    )

variant_metric_rows = []
for variant, profile in cluster_profiles.items():
    spec = cluster_solutions[variant]
    labels = np.asarray(cluster_variant_labels[variant])
    valid_labels = labels[labels >= 0]
    n_topics = int(len(np.unique(valid_labels)))
    outlier_count = int((labels == -1).sum())
    pairs = keyword_overlap_pairs[keyword_overlap_pairs['variant'].eq(variant)]
    flat_terms = [term for cid, terms_local in keyword_terms_by_variant[variant].items() if int(cid) != -1 for term in terms_local[:10]]
    term_counts = Counter(flat_terms)
    reused_terms = sorted([term for term, count in term_counts.items() if count > 1])
    diag_rows = diagnostics[(diagnostics['feature_space'].eq(spec['feature_space'])) & (diagnostics['k'].eq(selected_k))]
    if not diag_rows.empty:
        diag_row = diag_rows.iloc[0]
        silhouette_value = float(diag_row.get('silhouette', np.nan))
        db_value = float(diag_row.get('davies_bouldin', np.nan))
        ch_value = float(diag_row.get('calinski_harabasz', np.nan))
    else:
        silhouette_value, db_value, ch_value = embedding_cluster_scores(labels)
    topic_rows = profile[profile['cluster_id'].ge(0)]
    min_cluster_size = int(topic_rows['conteo'].min()) if not topic_rows.empty else 0
    max_cluster_size = int(topic_rows['conteo'].max()) if not topic_rows.empty else 0
    largest_topic_share = max_cluster_size / max(1, int(topic_rows['conteo'].sum())) if not topic_rows.empty else np.nan
    singleton_topics = int(topic_rows['conteo'].le(1).sum()) if not topic_rows.empty else 0
    topic_coherence = coherence_score(keyword_terms_by_variant[variant], coherence='c_v')
    topic_coherence_umass = coherence_score(keyword_terms_by_variant[variant], coherence='u_mass')
    topic_coherence_uci = coherence_score(keyword_terms_by_variant[variant], coherence='c_uci')
    topic_coherence_npmi = coherence_score(keyword_terms_by_variant[variant], coherence='c_npmi')
    mean_jaccard = float(pairs['keyword_jaccard'].mean()) if not pairs.empty else np.nan
    topic_diversity = len(set(flat_terms)) / max(1, len(flat_terms))
    outlier_rate = outlier_count / len(labels)
    balance_factor = max(0.0, 1.0 - (largest_topic_share if pd.notna(largest_topic_share) else 1.0))
    singleton_factor = max(0.0, 1.0 - singleton_topics / max(1, n_topics))
    overlap_factor = 1.0 - (mean_jaccard if pd.notna(mean_jaccard) else 1.0)
    coherence_factor = topic_coherence if pd.notna(topic_coherence) else 0.0
    interpretability_score = coherence_factor * topic_diversity * overlap_factor * balance_factor * singleton_factor * (1.0 - min(outlier_rate, 0.5))
    variant_metric_rows.append({
        'variant': variant,
        'variant_label': spec['variant_label'],
        'feature_space': spec['feature_space'],
        'algorithm': spec.get('algorithm', 'kmeans'),
        'k': selected_k if variant in {'embeddings', 'umap_2d', 'umap_3d'} else n_topics,
        'n_topics': n_topics,
        'outlier_count': outlier_count,
        'outlier_rate': outlier_rate,
        'silhouette': silhouette_value,
        'davies_bouldin': db_value,
        'calinski_harabasz': ch_value,
        'language_nmi': float(normalized_mutual_info_score(language_labels, labels)) if language_labels.nunique() > 1 else np.nan,
        'program_nmi': float(normalized_mutual_info_score(program_labels, labels)) if program_labels.nunique() > 1 else np.nan,
        'level_nmi': float(normalized_mutual_info_score(level_labels, labels)) if level_labels.nunique() > 1 else np.nan,
        'topic_coherence_cv': topic_coherence,
        'topic_coherence_umass': topic_coherence_umass,
        'topic_coherence_uci': topic_coherence_uci,
        'topic_coherence_npmi': topic_coherence_npmi,
        'topic_diversity': topic_diversity,
        'mean_keyword_jaccard': mean_jaccard,
        'max_keyword_jaccard': float(pairs['keyword_jaccard'].max()) if not pairs.empty else np.nan,
        'mean_shared_keywords': float(pairs['n_shared_keywords'].mean()) if not pairs.empty else np.nan,
        'max_shared_keywords': int(pairs['n_shared_keywords'].max()) if not pairs.empty else 0,
        'reused_top_keywords': len(reused_terms),
        'reused_top_keyword_rate': len(reused_terms) / max(1, len(term_counts)),
        'reused_keyword_examples': ', '.join(reused_terms[:12]),
        'min_cluster_size': min_cluster_size,
        'max_cluster_size': max_cluster_size,
        'largest_topic_share': largest_topic_share,
        'singleton_topics': singleton_topics,
        'interpretability_score': interpretability_score,
        'description': spec['description'],
    })

cluster_variant_metrics = pd.DataFrame(variant_metric_rows)


def safe_nmi(left, right):
    left = np.asarray(left)
    right = np.asarray(right)
    if len(left) < 3 or len(np.unique(left)) < 2 or len(np.unique(right)) < 2:
        return np.nan
    return float(normalized_mutual_info_score(left, right))


def top2_margin(matrix):
    arr = np.asarray(matrix, dtype=float)
    if arr.ndim != 2 or arr.shape[1] < 2:
        return np.nan
    totals = arr.sum(axis=1, keepdims=True)
    normalized_arr = np.divide(arr, totals, out=np.zeros_like(arr), where=totals > 0)
    ordered = np.sort(normalized_arr, axis=1)
    return float(np.nanmean(ordered[:, -1] - ordered[:, -2]))


def model_assignment_margin(variant):
    if variant == 'nmf_tfidf':
        return top2_margin(nmf_doc_topic)
    if variant == 'lda_count':
        return top2_margin(lda_doc_topic)
    if variant == 'bertopic_hdbscan':
        probs = getattr(bertopic_model.hdbscan_model, 'probabilities_', None)
        return float(np.nanmean(probs)) if probs is not None and len(probs) else np.nan
    if variant == 'consensus_ensemble':
        return float(df_layout['consensus_membership_margin'].mean())
    model = cluster_variant_models.get(variant)
    spec = cluster_solutions.get(variant, {})
    features = spec.get('features')
    if model is not None and hasattr(model, 'transform') and features is not None and spec.get('feature_space') != 'consensus_coassociation':
        distances = np.asarray(model.transform(features), dtype=float)
        if distances.ndim == 2 and distances.shape[1] >= 2:
            ordered = np.sort(distances, axis=1)
            return float(np.nanmean(1.0 - ordered[:, 0] / np.maximum(ordered[:, 1], 1e-12)))
    return np.nan


def recluster_variant_on_sample(variant, sample_idx, seed):
    spec = cluster_solutions[variant]
    baseline = np.asarray(cluster_variant_labels[variant])
    n_clusters = len(np.unique(baseline[baseline >= 0]))
    if n_clusters < 2:
        return None
    if variant == 'consensus_ensemble':
        sample_distance = 1.0 - consensus_similarity[np.ix_(sample_idx, sample_idx)]
        sample_distance = (sample_distance + sample_distance.T) / 2
        np.fill_diagonal(sample_distance, 0.0)
        return AgglomerativeClustering(n_clusters=n_clusters, metric='precomputed', linkage='average').fit_predict(sample_distance)
    if variant == 'bertopic_hdbscan':
        return None
    features = spec.get('features')
    if features is None or spec.get('feature_space') == 'consensus_coassociation':
        return None
    sample_features = features[sample_idx]
    algorithm = spec.get('algorithm', 'kmeans')
    if algorithm == 'agglomerative_ward':
        return AgglomerativeClustering(n_clusters=n_clusters, metric='euclidean', linkage='ward').fit_predict(sample_features)
    if algorithm in {'nmf_tfidf', 'lda_count'}:
        return None
    return KMeans(n_clusters=n_clusters, random_state=seed, n_init=30).fit_predict(sample_features)


def full_seed_stability(variant):
    spec = cluster_solutions[variant]
    baseline = np.asarray(cluster_variant_labels[variant])
    n_clusters = len(np.unique(baseline[baseline >= 0]))
    if n_clusters < 2:
        return np.nan
    if variant == 'consensus_ensemble':
        scores = []
        for source_variant in CONSENSUS_VARIANTS:
            source_labels = np.asarray(cluster_variant_labels[source_variant])
            valid = source_labels >= 0
            scores.append(safe_nmi(baseline[valid], source_labels[valid]))
        return float(np.nanmean(scores)) if scores else np.nan
    if spec.get('algorithm') == 'agglomerative_ward':
        return 1.0
    if spec.get('algorithm') in {'bertopic_hdbscan', 'nmf_tfidf', 'lda_count'}:
        return np.nan
    features = spec.get('features')
    if features is None:
        return np.nan
    scores = []
    for offset in range(max(3, min(STABILITY_BOOTSTRAPS, 10))):
        seed = RANDOM_STATE + 101 + offset
        candidate = KMeans(n_clusters=n_clusters, random_state=seed, n_init=30).fit_predict(features)
        scores.append(safe_nmi(baseline, candidate))
    return float(np.nanmean(scores)) if scores else np.nan

rng = np.random.default_rng(RANDOM_STATE)
stability_rows = []
for variant in cluster_solutions:
    baseline = np.asarray(cluster_variant_labels[variant])
    sample_scores = []
    sample_size = max(20, int(len(df_layout) * 0.82))
    for offset in range(max(3, STABILITY_BOOTSTRAPS)):
        sample_idx = np.sort(rng.choice(len(df_layout), size=sample_size, replace=False))
        sample_labels = recluster_variant_on_sample(variant, sample_idx, RANDOM_STATE + 301 + offset)
        if sample_labels is not None:
            base_sample = baseline[sample_idx]
            valid = base_sample >= 0
            sample_scores.append(safe_nmi(base_sample[valid], np.asarray(sample_labels)[valid]))
    seed_nmi = full_seed_stability(variant)
    bootstrap_nmi = float(np.nanmean(sample_scores)) if sample_scores else np.nan
    assignment_margin = model_assignment_margin(variant)
    components = [value for value in [seed_nmi, bootstrap_nmi, assignment_margin] if pd.notna(value)]
    stability_score = float(np.nanmean(components)) if components else np.nan
    stability_rows.append({
        'variant': variant,
        'seed_stability_nmi': seed_nmi,
        'bootstrap_stability_nmi': bootstrap_nmi,
        'assignment_margin_mean': assignment_margin,
        'stability_score': stability_score,
        'stability_bootstraps': int(STABILITY_BOOTSTRAPS),
    })

topic_stability = pd.DataFrame(stability_rows)
cluster_variant_metrics = cluster_variant_metrics.merge(topic_stability, on='variant', how='left')
cluster_variant_metrics['robust_interpretability_score'] = cluster_variant_metrics['interpretability_score'] * (
    0.5 + 0.5 * cluster_variant_metrics['stability_score'].fillna(cluster_variant_metrics['stability_score'].median())
)

objective_columns = [
    ('topic_coherence_cv', 'max'),
    ('mean_keyword_jaccard', 'min'),
    ('largest_topic_share', 'min'),
    ('outlier_rate', 'min'),
    ('stability_score', 'max'),
    ('language_nmi', 'min'),
]


def dominates(row_a, row_b, objectives):
    better_or_equal = True
    strictly_better = False
    for col, direction in objectives:
        a = row_a[col]
        b = row_b[col]
        if pd.isna(a):
            a = -np.inf if direction == 'max' else np.inf
        if pd.isna(b):
            b = -np.inf if direction == 'max' else np.inf
        if direction == 'max':
            better_or_equal &= a >= b
            strictly_better |= a > b
        else:
            better_or_equal &= a <= b
            strictly_better |= a < b
    return bool(better_or_equal and strictly_better)

pareto_flags = []
for idx, row in cluster_variant_metrics.iterrows():
    dominated = False
    for other_idx, other in cluster_variant_metrics.iterrows():
        if idx == other_idx:
            continue
        if dominates(other, row, objective_columns):
            dominated = True
            break
    pareto_flags.append(not dominated)

cluster_variant_metrics['pareto_frontier'] = pareto_flags
cluster_variant_metrics = (
    cluster_variant_metrics
    .sort_values(['robust_interpretability_score', 'interpretability_score', 'topic_coherence_cv', 'mean_keyword_jaccard'], ascending=[False, False, False, True])
    .reset_index(drop=True)
)
topic_model_frontier = cluster_variant_metrics[cluster_variant_metrics['pareto_frontier']].copy().reset_index(drop=True)
topic_model_metrics = cluster_variant_metrics.copy()

print('Comparación de variantes: menor traslape de keywords y mayor coherencia/diversidad son mejores.')
display(cluster_variant_metrics[[
    'variant_label', 'n_topics', 'outlier_rate', 'silhouette', 'language_nmi', 'program_nmi', 'level_nmi', 'topic_coherence_cv',
    'topic_coherence_npmi', 'topic_diversity', 'mean_keyword_jaccard', 'max_keyword_jaccard', 'max_shared_keywords',
    'min_cluster_size', 'max_cluster_size', 'largest_topic_share', 'singleton_topics', 'stability_score',
    'pareto_frontier', 'interpretability_score', 'robust_interpretability_score'
]])

display(cluster_overview[['cluster_label', 'cluster_theme', 'conteo', 'anio_promedio', 'idiomas', 'keywords', 'tesis_representativas']])


## Mapa semántico interactivo


In [ ]:
hover_cols = {
    'cluster_label': True,
    'cluster_theme': False,
    'titulo': True,
    'autorx': True,
    'anio_pub': True,
    'nivel': True,
    'programa': True,
    'asesor_unificado': True,
    'idioma': True,
    'periodo': True,
    'url': True,
    'umap_x': False,
    'umap_y': False,
}

fig_map = px.scatter(
    df_layout,
    x='umap_x',
    y='umap_y',
    color='cluster_theme',
    category_orders={'cluster_theme': cluster_theme_order},
    color_discrete_map=cluster_theme_color_map,
    hover_name='titulo',
    hover_data=hover_cols,
    title='Mapa semántico de todas las tesis del CIDE',
    width=1100,
    height=740,
)
fig_map.update_traces(marker=dict(size=8.5, opacity=0.84, line=dict(width=0.45, color='white')))
fig_map.update_layout(legend_title_text='Tema', margin=dict(l=30, r=30, t=70, b=30), plot_bgcolor='white')
fig_map.update_xaxes(showticklabels=False, title='', showgrid=False, zeroline=False)
fig_map.update_yaxes(showticklabels=False, title='', showgrid=False, zeroline=False)
fig_map.show()


## Mapa semántico 3D exploratorio

La vista 3D usa la misma reducción UMAP con tres componentes. No reemplaza al mapa 2D ni al clustering: sirve para inspeccionar vecindades, puentes y separaciones que pueden quedar comprimidas en el plano.


In [ ]:
hover_cols_3d = {
    'cluster_label': True,
    'cluster_theme': False,
    'titulo': True,
    'autorx': True,
    'anio_pub': True,
    'nivel': True,
    'programa': True,
    'asesor_unificado': True,
    'idioma': True,
    'periodo': True,
    'url': True,
    'umap_x': False,
    'umap_y': False,
    'umap_z': False,
}

fig_map_3d = px.scatter_3d(
    df_layout,
    x='umap_x',
    y='umap_y',
    z='umap_z',
    color='cluster_theme',
    category_orders={'cluster_theme': cluster_theme_order},
    color_discrete_map=cluster_theme_color_map,
    hover_name='titulo',
    hover_data=hover_cols_3d,
    title='Mapa semántico 3D exploratorio de tesis CIDE',
    width=1080,
    height=760,
)
fig_map_3d.update_traces(marker=dict(size=4.2, opacity=0.82, line=dict(width=0)))
fig_map_3d.update_layout(
    legend_title_text='Tema',
    margin=dict(l=10, r=10, t=70, b=10),
    scene=dict(
        xaxis=dict(visible=False, showbackground=False),
        yaxis=dict(visible=False, showbackground=False),
        zaxis=dict(visible=False, showbackground=False),
        bgcolor='white',
        camera=dict(eye=dict(x=1.65, y=1.65, z=1.15)),
    ),
    paper_bgcolor='white',
)
fig_map_3d.show()


## Comparar KMeans sobre UMAP 2D y UMAP 3D


In [ ]:
variant_hover_cols = {
    'titulo': True,
    'anio_pub': True,
    'nivel': True,
    'programa': True,
    'idioma': True,
    'asesor_unificado': True,
    'cluster_theme_umap_2d': False,
    'cluster_theme_umap_3d': False,
    'cluster_theme_bertopic_hdbscan': False,
    'resumen': False,
}

fig_map_variant_2d = px.scatter(
    df_layout,
    x='umap_x',
    y='umap_y',
    color='cluster_theme_umap_2d',
    category_orders={'cluster_theme_umap_2d': variant_theme_orders['umap_2d']},
    color_discrete_map=variant_theme_color_maps['umap_2d'],
    hover_name='titulo',
    hover_data=variant_hover_cols,
    title='KMeans calculado sobre la proyección UMAP 2D',
    width=980,
    height=620,
    labels={'cluster_theme_umap_2d': 'Tema experimental', 'umap_x': 'UMAP 1', 'umap_y': 'UMAP 2'},
)
fig_map_variant_2d.update_traces(marker=dict(size=9, opacity=0.84, line=dict(width=0.4, color='white')))
fig_map_variant_2d.update_layout(legend_title_text='Tema experimental', margin=dict(l=30, r=20, t=70, b=30))
fig_map_variant_2d.show()

fig_map_variant_3d = px.scatter_3d(
    df_layout,
    x='umap_x',
    y='umap_y',
    z='umap_z',
    color='cluster_theme_umap_3d',
    category_orders={'cluster_theme_umap_3d': variant_theme_orders['umap_3d']},
    color_discrete_map=variant_theme_color_maps['umap_3d'],
    hover_name='titulo',
    hover_data=variant_hover_cols,
    title='KMeans calculado sobre la proyección UMAP 3D',
    width=980,
    height=680,
    labels={
        'cluster_theme_umap_3d': 'Tema experimental',
        'umap_x': 'UMAP 1',
        'umap_y': 'UMAP 2',
        'umap_z': 'UMAP 3',
    },
)
fig_map_variant_3d.update_traces(marker=dict(size=4.8, opacity=0.82, line=dict(width=0.2, color='white')))
fig_map_variant_3d.update_layout(legend_title_text='Tema experimental', margin=dict(l=0, r=0, t=70, b=0))
fig_map_variant_3d.show()

fig_map_bertopic = px.scatter(
    df_layout,
    x='umap_x',
    y='umap_y',
    color='cluster_theme_bertopic_hdbscan',
    category_orders={'cluster_theme_bertopic_hdbscan': variant_theme_orders['bertopic_hdbscan']},
    color_discrete_map=variant_theme_color_maps['bertopic_hdbscan'],
    hover_name='titulo',
    hover_data=variant_hover_cols,
    title='BERTopic + HDBSCAN: tópicos c-TF-IDF sobre el mapa 2D',
    width=980,
    height=640,
    labels={'cluster_theme_bertopic_hdbscan': 'Tópico', 'umap_x': 'UMAP 1', 'umap_y': 'UMAP 2'},
)
fig_map_bertopic.update_traces(marker=dict(size=9, opacity=0.84, line=dict(width=0.4, color='white')))
fig_map_bertopic.update_layout(legend_title_text='Tópico', margin=dict(l=30, r=20, t=70, b=30))
fig_map_bertopic.show()

variant_metric_long = cluster_variant_metrics.melt(
    id_vars=['variant', 'variant_label'],
    value_vars=['mean_keyword_jaccard', 'max_keyword_jaccard'],
    var_name='metric',
    value_name='value',
)
variant_metric_long['metric_label'] = variant_metric_long['metric'].map({
    'mean_keyword_jaccard': 'Traslape promedio',
    'max_keyword_jaccard': 'Peor par de clusters',
})
fig_variant_overlap = px.bar(
    variant_metric_long,
    x='variant_label',
    y='value',
    color='metric_label',
    barmode='group',
    title='Traslape de keywords entre clusters/tópicos por variante',
    width=960,
    height=440,
    labels={'variant_label': 'Variante', 'value': 'Jaccard de keywords top-10', 'metric_label': 'Métrica'},
    color_discrete_map={'Traslape promedio': '#2563eb', 'Peor par de clusters': '#d97706'},
)
fig_variant_overlap.update_layout(margin=dict(l=40, r=20, t=70, b=95), legend_title_text='Métrica')
fig_variant_overlap.show()

fig_topic_quality = px.scatter(
    cluster_variant_metrics,
    x='mean_keyword_jaccard',
    y='topic_coherence_cv',
    size='n_topics',
    color='variant_label',
    hover_data=['topic_diversity', 'outlier_rate', 'max_keyword_jaccard', 'reused_top_keywords'],
    title='Calidad interpretativa: coherencia vs traslape de keywords',
    width=920,
    height=470,
    labels={
        'mean_keyword_jaccard': 'Traslape promedio de keywords (menor es mejor)',
        'topic_coherence_cv': 'Coherencia c_v (mayor es mejor)',
        'variant_label': 'Variante',
        'n_topics': 'Temas',
    },
)
fig_topic_quality.update_traces(marker=dict(opacity=0.85, line=dict(width=0.8, color='white')))
fig_topic_quality.update_layout(margin=dict(l=55, r=25, t=70, b=55), legend_title_text='Variante')
fig_topic_quality.show()


fig_map_consensus = px.scatter(
    df_layout,
    x='umap_x',
    y='umap_y',
    color='cluster_theme_consensus_ensemble',
    category_orders={'cluster_theme_consensus_ensemble': variant_theme_orders['consensus_ensemble']},
    color_discrete_map=variant_theme_color_maps['consensus_ensemble'],
    hover_name='titulo',
    hover_data={
        'anio_pub': True,
        'nivel': True,
        'programa': True,
        'idioma': True,
        'asesor_unificado': True,
        'macro_cluster_theme': True,
        'consensus_membership_margin': ':.3f',
        'resumen': False,
    },
    title='Consenso entre modelos: comunidades estables sobre el mapa semantico',
    width=980,
    height=640,
    labels={'cluster_theme_consensus_ensemble': 'Subtema consenso', 'umap_x': 'UMAP 1', 'umap_y': 'UMAP 2'},
)
fig_map_consensus.update_traces(marker=dict(size=9, opacity=0.86, line=dict(width=0.4, color='white')))
fig_map_consensus.update_layout(legend_title_text='Subtema consenso', margin=dict(l=30, r=20, t=70, b=30))
fig_map_consensus.show()

frontier_plot = cluster_variant_metrics.copy()
frontier_plot['frontier_label'] = np.where(frontier_plot['pareto_frontier'], 'Frontera Pareto', 'Dominado')
fig_topic_frontier = px.scatter(
    frontier_plot,
    x='mean_keyword_jaccard',
    y='topic_coherence_cv',
    size='stability_score',
    color='frontier_label',
    symbol='algorithm',
    hover_name='variant_label',
    hover_data=['topic_coherence_npmi', 'largest_topic_share', 'outlier_rate', 'language_nmi', 'robust_interpretability_score'],
    title='Frontera Pareto: coherencia, bajo traslape y estabilidad',
    width=920,
    height=500,
    labels={
        'mean_keyword_jaccard': 'Traslape promedio de keywords (menor es mejor)',
        'topic_coherence_cv': 'Coherencia c_v (mayor es mejor)',
        'frontier_label': 'Estado',
        'stability_score': 'Estabilidad',
    },
    color_discrete_map={'Frontera Pareto': '#2563eb', 'Dominado': '#94a3b8'},
)
fig_topic_frontier.update_traces(marker=dict(opacity=0.88, line=dict(width=0.8, color='white')))
fig_topic_frontier.update_layout(margin=dict(l=55, r=25, t=70, b=55), legend_title_text='Estado')
fig_topic_frontier.show()

stability_plot = cluster_variant_metrics.sort_values('stability_score', ascending=True).copy()
stability_plot['frontier_label'] = np.where(stability_plot['pareto_frontier'], 'Frontera Pareto', 'Dominado')
fig_topic_stability = px.bar(
    stability_plot,
    x='stability_score',
    y='variant_label',
    color='frontier_label',
    orientation='h',
    title='Estabilidad de asignaciones por variante',
    width=900,
    height=520,
    labels={'stability_score': 'Score de estabilidad', 'variant_label': 'Variante', 'frontier_label': 'Estado'},
    color_discrete_map={'Frontera Pareto': '#2563eb', 'Dominado': '#94a3b8'},
)
fig_topic_stability.update_layout(margin=dict(l=210, r=20, t=70, b=45), legend_title_text='Estado')
fig_topic_stability.show()

fig_topic_taxonomy = px.sunburst(
    topic_taxonomy,
    path=['macro_theme', 'subtopic_theme'],
    values='conteo',
    color='macro_theme',
    color_discrete_map=cluster_theme_color_map,
    title='Taxonomia jerarquica: macrotemas y subtemas de consenso',
    width=880,
    height=640,
)
fig_topic_taxonomy.update_layout(margin=dict(l=10, r=10, t=70, b=10))
fig_topic_taxonomy.show()

fig_membership_ambiguity = px.histogram(
    df_layout,
    x='consensus_membership_margin',
    nbins=24,
    title='Ambiguedad tematica: diferencia entre primer y segundo subtema',
    width=880,
    height=440,
    labels={'consensus_membership_margin': 'Margen de membresia consenso', 'count': 'Tesis'},
    color_discrete_sequence=['#d97706'],
)
fig_membership_ambiguity.update_layout(margin=dict(l=55, r=20, t=70, b=55), bargap=0.04)
fig_membership_ambiguity.show()

worst_keyword_overlaps = (
    keyword_overlap_pairs.sort_values(['keyword_jaccard', 'n_shared_keywords'], ascending=[False, False])
    .head(12)
    .reset_index(drop=True)
)
display(worst_keyword_overlaps[[
    'variant_label', 'cluster_a', 'cluster_b', 'cluster_a_theme', 'cluster_b_theme',
    'n_shared_keywords', 'keyword_jaccard', 'shared_keywords'
]])


## Máquina temporal del mapa semántico

Esta vista usa una geometría UMAP fija y anima la aparición de tesis por año. Los puntos grises son tesis publicadas antes del año activo; los puntos de color son las tesis nuevas del año, codificadas por tema. Así la película muestra cómo se llena el espacio semántico sin hacer que los puntos “se muevan” artificialmente.


In [ ]:
year_values = df_layout['anio_pub'].dropna().astype(int)
timeline_years = sorted(year_values.unique().tolist())

if not timeline_years:
    fig_time_machine = None
    fig_period_facets = None
    print('No hay años de publicación suficientes para construir la vista temporal.')
else:
    x_range = [df_layout['umap_x'].min() - 0.7, df_layout['umap_x'].max() + 0.7]
    y_range = [df_layout['umap_y'].min() - 0.7, df_layout['umap_y'].max() + 0.7]

    def temporal_customdata(frame):
        if frame.empty:
            return np.empty((0, 6))
        return frame[['titulo', 'autorx', 'asesor_unificado', 'anio_pub', 'idioma', 'url']].fillna('').to_numpy()

    def movie_title(year):
        accumulated = int(df_layout['anio_pub'].astype('Int64').le(year).sum())
        previous = int(df_layout['anio_pub'].astype('Int64').lt(year).sum())
        new = int(df_layout['anio_pub'].astype('Int64').eq(year).sum())
        active_themes = int(df_layout.loc[df_layout['anio_pub'].astype('Int64').eq(year), 'cluster_theme'].nunique())
        return f'Mapa semántico acumulado hasta {year} · {previous} previas · {new} nuevas · {active_themes} temas activos'

    def cumulative_traces_for_year(year, showlegend=False):
        history = df_layout[df_layout['anio_pub'].astype('Int64').lt(year)]
        current = df_layout[df_layout['anio_pub'].astype('Int64').eq(year)]
        traces = [
            go.Scatter(
                x=history['umap_x'],
                y=history['umap_y'],
                mode='markers',
                name='Tesis previas',
                marker=dict(size=5.6, color='rgba(100,116,139,0.28)', line=dict(width=0)),
                customdata=temporal_customdata(history),
                hovertemplate=(
                    '<b>%{customdata[0]}</b><br>'
                    'Autor/a: %{customdata[1]}<br>'
                    'Asesor/a: %{customdata[2]}<br>'
                    'Año tesis: %{customdata[3]}<br>'
                    'Idioma: %{customdata[4]}<br>'
                    '<extra>Tesis previa</extra>'
                ),
                showlegend=showlegend,
            )
        ]

        for theme in cluster_theme_order:
            subset = current[current['cluster_theme'].eq(theme)]
            traces.append(
                go.Scatter(
                    x=subset['umap_x'],
                    y=subset['umap_y'],
                    mode='markers',
                    name=theme,
                    legendgroup=theme,
                    showlegend=showlegend,
                    marker=dict(
                        size=12.5,
                        color=cluster_theme_color_map.get(theme, '#2563eb'),
                        opacity=0.96,
                        line=dict(width=0.85, color='white'),
                    ),
                    customdata=temporal_customdata(subset),
                    hovertemplate=(
                        '<b>%{customdata[0]}</b><br>'
                        'Autor/a: %{customdata[1]}<br>'
                        'Asesor/a: %{customdata[2]}<br>'
                        'Año tesis: %{customdata[3]}<br>'
                        'Idioma: %{customdata[4]}<br>'
                        '<extra>Tesis nueva</extra>'
                    ),
                )
            )
        return traces

    first_year = timeline_years[0]
    frame_trace_ids = list(range(1 + len(cluster_theme_order)))
    frames = [
        go.Frame(
            name=str(year),
            data=cumulative_traces_for_year(year, showlegend=False),
            traces=frame_trace_ids,
            layout=go.Layout(title_text=movie_title(year)),
        )
        for year in timeline_years
    ]

    fig_time_machine = go.Figure(data=cumulative_traces_for_year(first_year, showlegend=True), frames=frames)
    slider_steps = [
        dict(
            method='animate',
            label=str(year),
            args=[
                [str(year)],
                dict(mode='immediate', frame=dict(duration=560, redraw=True), transition=dict(duration=260, easing='cubic-in-out')),
            ],
        )
        for year in timeline_years
    ]
    fig_time_machine.update_layout(
        title=movie_title(first_year),
        width=1120,
        height=770,
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend_title_text='Tema nuevo del año',
        margin=dict(l=25, r=25, t=78, b=76),
        xaxis=dict(range=x_range, visible=False),
        yaxis=dict(range=y_range, visible=False, scaleanchor='x', scaleratio=1),
        updatemenus=[
            dict(
                type='buttons',
                direction='left',
                x=0,
                y=-0.055,
                xanchor='left',
                yanchor='top',
                buttons=[
                    dict(
                        label='Play',
                        method='animate',
                        args=[None, dict(frame=dict(duration=820, redraw=True), fromcurrent=True, transition=dict(duration=320, easing='cubic-in-out'))],
                    ),
                    dict(
                        label='Pause',
                        method='animate',
                        args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate', transition=dict(duration=0))],
                    ),
                ],
            )
        ],
        sliders=[
            dict(
                active=0,
                currentvalue=dict(prefix='Año ', font=dict(size=13)),
                pad=dict(t=44, b=5),
                steps=slider_steps,
            )
        ],
    )
    fig_time_machine.show()

    period_order = [p for p in ['1990s', '2000s', '2010s', '2020s'] if p in set(df_layout['periodo'].dropna())]
    fig_period_facets = px.scatter(
        df_layout.dropna(subset=['periodo']),
        x='umap_x',
        y='umap_y',
        color='cluster_theme',
        facet_col='periodo',
        facet_col_wrap=2,
        category_orders={'cluster_theme': cluster_theme_order, 'periodo': period_order},
        color_discrete_map=cluster_theme_color_map,
        hover_name='titulo',
        hover_data=hover_cols,
        title='Mapa semántico por periodo de publicación',
        width=1120,
        height=720,
    )
    fig_period_facets.update_traces(marker=dict(size=7.2, opacity=0.76, line=dict(width=0.35, color='white')))
    fig_period_facets.for_each_annotation(lambda a: a.update(text=a.text.replace('periodo=', '')))
    fig_period_facets.update_layout(legend_title_text='Tema', margin=dict(l=20, r=20, t=72, b=30), plot_bgcolor='white')
    fig_period_facets.update_xaxes(showticklabels=False, title='', showgrid=False, zeroline=False, matches=None)
    fig_period_facets.update_yaxes(showticklabels=False, title='', showgrid=False, zeroline=False, matches=None)
    fig_period_facets.show()


## Evolución temporal de temas

Se genera una grilla completa año × tema para ver tanto presencia como ausencia. El heatmap ayuda a detectar picos puntuales; el streamgraph muestra cómo cambia la mezcla temática de cada cohorte anual.

In [ ]:

cluster_year_observed = (
    df_layout.dropna(subset=['anio_pub'])
    .assign(anio_pub=lambda d: d['anio_pub'].astype(int))
    .groupby(['anio_pub', 'cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label'])
    .size()
    .reset_index(name='conteo')
)

all_years = list(range(int(df_layout['anio_pub'].min()), int(df_layout['anio_pub'].max()) + 1))
cluster_meta = (
    df_layout[['cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label']]
    .drop_duplicates()
    .sort_values('cluster_id')
)
cluster_year_grid = (
    pd.MultiIndex.from_product([all_years, cluster_order], names=['anio_pub', 'cluster_label'])
    .to_frame(index=False)
    .merge(cluster_meta, on='cluster_label', how='left')
    .merge(
        cluster_year_observed[['anio_pub', 'cluster_label', 'conteo']],
        on=['anio_pub', 'cluster_label'],
        how='left',
    )
)
cluster_year_grid['conteo'] = cluster_year_grid['conteo'].fillna(0).astype(int)
year_totals = df_layout.dropna(subset=['anio_pub']).assign(anio_pub=lambda d: d['anio_pub'].astype(int)).groupby('anio_pub').size()
cluster_year_grid['total_anio'] = cluster_year_grid['anio_pub'].map(year_totals).fillna(0).astype(int)
cluster_year_grid['participacion_anio'] = np.where(
    cluster_year_grid['total_anio'] > 0,
    cluster_year_grid['conteo'] / cluster_year_grid['total_anio'],
    0.0,
)
cluster_year = cluster_year_grid.copy()
cluster_year.to_parquet(CLUSTER_YEAR_PATH, index=False)

heatmap_data = cluster_year.pivot_table(
    index='cluster_theme',
    columns='anio_pub',
    values='participacion_anio',
    fill_value=0,
).reindex(cluster_theme_order)

fig_year_heatmap = px.imshow(
    heatmap_data,
    aspect='auto',
    color_continuous_scale='Blues',
    title='Participación de cada tema por año',
    labels=dict(x='Año', y='Tema', color='Participación'),
    width=1060,
    height=620,
)
fig_year_heatmap.update_layout(margin=dict(l=210, r=30, t=70, b=50))
fig_year_heatmap.show()

fig_theme_stream = go.Figure()
for cluster_name in cluster_order:
    subset = cluster_year[cluster_year['cluster_label'].eq(cluster_name)].sort_values('anio_pub')
    theme = cluster_label_to_theme.get(cluster_name, cluster_name)
    fig_theme_stream.add_trace(
        go.Scatter(
            x=subset['anio_pub'],
            y=subset['participacion_anio'] * 100,
            mode='lines',
            stackgroup='one',
            name=theme,
            line=dict(width=0.5, color=cluster_color_map.get(cluster_name, '#2563eb')),
            hovertemplate=f'{theme}<br>Año: %{{x}}<br>Participación: %{{y:.1f}}%<extra></extra>',
        )
    )
fig_theme_stream.update_layout(
    title='Composición temática anual de las tesis',
    width=1060,
    height=600,
    yaxis=dict(title='Participación anual (%)', range=[0, 100], ticksuffix='%'),
    xaxis=dict(title='Año', dtick=2),
    legend_title_text='Tema',
    margin=dict(l=45, r=25, t=70, b=45),
    plot_bgcolor='white',
    paper_bgcolor='white',
)
fig_theme_stream.show()


def race_trace_for_year(year):
    subset = (
        cluster_year[cluster_year['anio_pub'].eq(year)]
        .assign(participacion_pct=lambda d: d['participacion_anio'] * 100)
        .sort_values('participacion_pct', ascending=True)
    )
    labels = subset['cluster_theme'].tolist()
    colors = subset['cluster_label'].map(cluster_color_map).tolist()
    return go.Bar(
        x=subset['participacion_pct'],
        y=labels,
        orientation='h',
        marker=dict(color=colors, line=dict(color='white', width=0.6)),
        text=subset['conteo'].map(lambda x: f'{int(x)}'),
        textposition='outside',
        cliponaxis=False,
        hovertemplate='%{y}<br>Participación: %{x:.1f}%<br>Tesis: %{text}<extra></extra>',
    )

def race_title(year):
    total = int(cluster_year.loc[cluster_year['anio_pub'].eq(year), 'total_anio'].max())
    return f'Temas dominantes en {year} · {total} tesis'

race_years = [year for year in all_years if int(year_totals.get(year, 0)) > 0]
if race_years:
    first_race_year = race_years[0]
    fig_theme_race = go.Figure(data=[race_trace_for_year(first_race_year)])
    fig_theme_race.frames = [
        go.Frame(name=str(year), data=[race_trace_for_year(year)], layout=go.Layout(title_text=race_title(year)))
        for year in race_years
    ]
    race_steps = [
        dict(
            method='animate',
            label=str(year),
            args=[[str(year)], dict(mode='immediate', frame=dict(duration=520, redraw=True), transition=dict(duration=180))],
        )
        for year in race_years
    ]
    fig_theme_race.update_layout(
        title=race_title(first_race_year),
        width=1060,
        height=600,
        xaxis=dict(title='Participación anual (%)', range=[0, max(55, cluster_year['participacion_anio'].max() * 115)], ticksuffix='%'),
        yaxis=dict(title='', automargin=True),
        margin=dict(l=210, r=45, t=70, b=80),
        plot_bgcolor='white',
        paper_bgcolor='white',
        showlegend=False,
        updatemenus=[
            dict(
                type='buttons',
                direction='left',
                x=0,
                y=-0.12,
                xanchor='left',
                yanchor='top',
                buttons=[
                    dict(label='▶', method='animate', args=[None, dict(frame=dict(duration=760, redraw=True), fromcurrent=True, transition=dict(duration=220))]),
                    dict(label='II', method='animate', args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate', transition=dict(duration=0))]),
                ],
            )
        ],
        sliders=[dict(active=0, currentvalue=dict(prefix='Año ', font=dict(size=13)), pad=dict(t=48, b=5), steps=race_steps)],
    )
    fig_theme_race.show()
else:
    fig_theme_race = None

period_mix = (
    df_layout.dropna(subset=['periodo'])
    .groupby(['periodo', 'cluster_label'])
    .size()
    .reset_index(name='conteo_periodo')
)
period_mix['total_periodo'] = period_mix.groupby('periodo')['conteo_periodo'].transform('sum')
period_mix['participacion_periodo'] = period_mix['conteo_periodo'] / period_mix['total_periodo']
period_share = period_mix.pivot_table(
    index='cluster_label',
    columns='periodo',
    values='participacion_periodo',
    fill_value=0,
)

first_year = (
    cluster_year_observed.groupby('cluster_label')['anio_pub']
    .min()
    .rename('anio_inicio')
)
peak_rows = (
    cluster_year.sort_values(['cluster_label', 'conteo', 'participacion_anio', 'anio_pub'], ascending=[True, False, False, True])
    .groupby('cluster_label')
    .head(1)
    [['cluster_label', 'anio_pub', 'conteo', 'participacion_anio']]
    .rename(columns={'anio_pub': 'anio_pico', 'conteo': 'conteo_pico', 'participacion_anio': 'participacion_pico'})
)
cluster_lifecycle = (
    cluster_overview[['cluster_label', 'cluster_id', 'cluster_theme', 'conteo', 'keywords']]
    .merge(first_year, on='cluster_label', how='left')
    .merge(peak_rows, on='cluster_label', how='left')
    .sort_values('cluster_id')
)
cluster_lifecycle['participacion_pico'] = (cluster_lifecycle['participacion_pico'] * 100).round(1)
for period in ['2000s', '2010s', '2020s']:
    cluster_lifecycle[f'participacion_{period}'] = (
        cluster_lifecycle['cluster_label'].map(period_share.get(period, pd.Series(dtype=float))).fillna(0) * 100
    ).round(1)
cluster_lifecycle['cambio_2020s_vs_2010s_pp'] = (
    cluster_lifecycle.get('participacion_2020s', 0) - cluster_lifecycle.get('participacion_2010s', 0)
).round(1)
cluster_lifecycle.to_parquet(CLUSTER_LIFECYCLE_PATH, index=False)

display(cluster_lifecycle[['cluster_label', 'cluster_theme', 'conteo', 'anio_inicio', 'anio_pico', 'conteo_pico', 'participacion_pico', 'cambio_2020s_vs_2010s_pp']])


## Distribución de idioma por cluster

Esta tabla verifica si el modelo multilingüe está separando tesis por tema en vez de aislarlas por idioma. Si las tesis en inglés aparecen repartidas entre varios clusters, el espacio semántico está funcionando mejor para análisis bilingüe.


In [ ]:
if 'idioma' in df_layout.columns:
    cluster_language = (
        df_layout.assign(idioma=df_layout['idioma'].fillna('desconocido'))
        .groupby(['cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label', 'idioma'])
        .size()
        .reset_index(name='conteo')
    )
    cluster_language['total_cluster'] = cluster_language.groupby('cluster_label')['conteo'].transform('sum')
    cluster_language['participacion_cluster'] = cluster_language['conteo'] / cluster_language['total_cluster']
    cluster_language.to_parquet(CLUSTER_LANGUAGE_PATH, index=False)

    display(cluster_language.sort_values(['cluster_id', 'idioma']))

    fig_language = px.bar(
        cluster_language,
        x='cluster_theme',
        y='conteo',
        color='idioma',
        category_orders={'cluster_theme': cluster_theme_order},
        color_discrete_map=LANGUAGE_COLOR_MAP,
        title='Distribución de idioma por tema',
        width=1040,
        height=520,
    )
    fig_language.update_layout(
        barmode='stack', xaxis_title='Tema', yaxis_title='Número de tesis',
        legend_title_text='Idioma', margin=dict(l=50, r=30, t=70, b=165)
    )
    fig_language.update_xaxes(tickangle=35)
    fig_language.show()
else:
    cluster_language = pd.DataFrame(columns=['cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label', 'idioma', 'conteo', 'total_cluster', 'participacion_cluster'])
    cluster_language.to_parquet(CLUSTER_LANGUAGE_PATH, index=False)
    fig_language = None
    print('No hay columna idioma para evaluar distribución por cluster.')


## Programas, niveles e interdisciplinariedad

Este bloque distingue estructura temática de estructura administrativa. Calcula la composición de cada cluster por nivel y programa, entropía interdisciplinaria, similitud entre programas a partir de sus perfiles temáticos y evolución de la producción académica.


In [ ]:
level_order = [level for level in ['Licenciatura', 'Maestría', 'Doctorado'] if level in set(df_layout['nivel'])]
program_counts = (
    df_layout.groupby(['nivel', 'programa', 'grado_programa'])
    .size()
    .reset_index(name='n_tesis')
)
program_counts['nivel_order'] = program_counts['nivel'].map({level: i for i, level in enumerate(level_order)})
program_counts = program_counts.sort_values(['nivel_order', 'n_tesis', 'grado_programa'], ascending=[True, False, True])
program_order = program_counts['grado_programa'].tolist()
program_color_map = {name: PALETTE[i % len(PALETTE)] for i, name in enumerate(program_order)}

cluster_program = (
    df_layout.groupby([
        'cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label',
        'nivel', 'programa', 'grado_programa'
    ])
    .size()
    .reset_index(name='conteo')
)
cluster_program['total_cluster'] = cluster_program.groupby('cluster_label')['conteo'].transform('sum')
cluster_program['total_programa'] = cluster_program.groupby('grado_programa')['conteo'].transform('sum')
cluster_program['participacion_cluster'] = cluster_program['conteo'] / cluster_program['total_cluster']
cluster_program['participacion_programa'] = cluster_program['conteo'] / cluster_program['total_programa']

cluster_level = (
    df_layout.groupby(['cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label', 'nivel'])
    .size()
    .reset_index(name='conteo')
)
cluster_level['total_cluster'] = cluster_level.groupby('cluster_label')['conteo'].transform('sum')
cluster_level['participacion_cluster'] = cluster_level['conteo'] / cluster_level['total_cluster']


def normalized_entropy(counts, denominator_categories):
    values = np.asarray(counts, dtype=float)
    probabilities = values / values.sum()
    entropy = -np.sum(probabilities * np.log(np.maximum(probabilities, 1e-12)))
    return float(entropy / np.log(max(2, denominator_categories)))

interdisciplinarity_rows = []
for cluster_label, group in cluster_program.groupby('cluster_label'):
    group = group.sort_values('conteo', ascending=False)
    first = group.iloc[0]
    probabilities = group['conteo'].to_numpy(dtype=float) / group['conteo'].sum()
    level_group = cluster_level[cluster_level['cluster_label'].eq(cluster_label)].sort_values('conteo', ascending=False)
    interdisciplinarity_rows.append({
        'cluster_label': cluster_label,
        'cluster_id': int(first['cluster_id']),
        'cluster_theme': first['cluster_theme'],
        'cluster_display_label': first['cluster_display_label'],
        'conteo': int(group['conteo'].sum()),
        'n_programas': int(group['grado_programa'].nunique()),
        'n_niveles': int(group['nivel'].nunique()),
        'entropia_programas': normalized_entropy(group['conteo'], len(program_order)),
        'hhi_programas': float(np.sum(probabilities ** 2)),
        'programas_efectivos': float(np.exp(-np.sum(probabilities * np.log(np.maximum(probabilities, 1e-12))))),
        'programa_dominante': first['grado_programa'],
        'participacion_programa_dominante': float(first['participacion_cluster']),
        'nivel_dominante': level_group.iloc[0]['nivel'],
        'participacion_nivel_dominante': float(level_group.iloc[0]['participacion_cluster']),
        'programas_top': ' | '.join(
            f"{row.grado_programa} ({row.participacion_cluster:.1%})"
            for row in group.head(5).itertuples(index=False)
        ),
        'keywords': keywords_by_cluster.get(int(first['cluster_id']), ''),
    })

cluster_interdisciplinarity = (
    pd.DataFrame(interdisciplinarity_rows)
    .sort_values(['entropia_programas', 'conteo'], ascending=[False, False])
    .reset_index(drop=True)
)

program_summary_rows = []
for grade_program, group in cluster_program.groupby('grado_programa'):
    group = group.sort_values('participacion_programa', ascending=False)
    first = group.iloc[0]
    probabilities = group['conteo'].to_numpy(dtype=float) / group['conteo'].sum()
    program_summary_rows.append({
        'nivel': first['nivel'],
        'programa': first['programa'],
        'grado_programa': grade_program,
        'n_tesis': int(group['conteo'].sum()),
        'n_clusters': int(group['cluster_label'].nunique()),
        'entropia_tematica': normalized_entropy(group['conteo'], selected_k),
        'concentracion_tematica_hhi': float(np.sum(probabilities ** 2)),
        'cluster_principal': first['cluster_theme'],
        'cluster_principal_label': first['cluster_label'],
        'participacion_cluster_principal': float(first['participacion_programa']),
        'temas_top': ' | '.join(
            f"{row.cluster_theme} ({row.participacion_programa:.1%})"
            for row in group.head(5).itertuples(index=False)
        ),
    })
program_cluster_summary = (
    pd.DataFrame(program_summary_rows)
    .sort_values(['nivel', 'n_tesis'], ascending=[True, False])
    .reset_index(drop=True)
)

program_distribution = (
    cluster_program.pivot_table(
        index='grado_programa', columns='cluster_label', values='participacion_programa', fill_value=0
    )
    .reindex(index=program_order, columns=cluster_order, fill_value=0)
)
program_similarity_matrix = cosine_similarity(program_distribution.to_numpy())
program_similarity = pd.DataFrame(
    [
        {
            'programa_a': program_order[i],
            'programa_b': program_order[j],
            'similitud_coseno': float(program_similarity_matrix[i, j]),
            'n_tesis_a': int(program_counts.set_index('grado_programa').loc[program_order[i], 'n_tesis']),
            'n_tesis_b': int(program_counts.set_index('grado_programa').loc[program_order[j], 'n_tesis']),
        }
        for i in range(len(program_order))
        for j in range(len(program_order))
    ]
)
program_similarity_pairs = (
    program_similarity[
        program_similarity.apply(
            lambda row: program_order.index(row['programa_a']) < program_order.index(row['programa_b']), axis=1
        )
    ]
    .sort_values('similitud_coseno', ascending=False)
    .reset_index(drop=True)
)
reliable_program_similarity = program_similarity_pairs[
    program_similarity_pairs['n_tesis_a'].ge(10) & program_similarity_pairs['n_tesis_b'].ge(10)
].reset_index(drop=True)

program_year = (
    df_layout.dropna(subset=['anio_pub'])
    .assign(anio_pub=lambda d: d['anio_pub'].astype(int))
    .groupby(['anio_pub', 'nivel', 'programa', 'grado_programa'])
    .size()
    .reset_index(name='conteo')
)
level_year = program_year.groupby(['anio_pub', 'nivel'], as_index=False)['conteo'].sum()

fig_map_level = px.scatter(
    df_layout,
    x='umap_x', y='umap_y', color='nivel',
    category_orders={'nivel': level_order}, color_discrete_map=LEVEL_COLOR_MAP,
    hover_name='titulo',
    hover_data={'programa': True, 'anio_pub': True, 'cluster_theme': True, 'asesor_unificado': True, 'resumen': False},
    title='El mapa semántico visto por nivel académico', width=1040, height=650,
)
fig_map_level.update_traces(marker=dict(size=7.5, opacity=0.72, line=dict(width=0.35, color='white')))
fig_map_level.update_layout(legend_title_text='Nivel', margin=dict(l=25, r=25, t=70, b=25))
fig_map_level.update_xaxes(visible=False)
fig_map_level.update_yaxes(visible=False)
fig_map_level.show()

level_pivot = (
    cluster_level.pivot_table(index='nivel', columns='cluster_display_label', values='participacion_cluster', fill_value=0)
    .reindex(index=level_order, columns=cluster_display_order, fill_value=0)
)
fig_cluster_level = go.Figure()
for level in level_order:
    subset = cluster_level[cluster_level['nivel'].eq(level)].set_index('cluster_display_label')
    fig_cluster_level.add_bar(
        name=level,
        x=cluster_display_order,
        y=[float(subset['participacion_cluster'].get(label, 0)) for label in cluster_display_order],
        marker_color=LEVEL_COLOR_MAP.get(level, '#64748b'),
    )
fig_cluster_level.update_layout(
    barmode='stack', title='Composición de cada tema por nivel académico', width=1100, height=520,
    xaxis_title='Tema', yaxis_title='Participación dentro del tema',
    yaxis_tickformat='.0%', margin=dict(l=60, r=25, t=70, b=180), legend_title_text='Nivel'
)
fig_cluster_level.update_xaxes(tickangle=35)
fig_cluster_level.show()

program_heatmap = (
    cluster_program.pivot_table(
        index='grado_programa', columns='cluster_display_label', values='participacion_programa', fill_value=0
    )
    .reindex(index=program_order, columns=cluster_display_order, fill_value=0)
)
fig_program_cluster_heatmap = go.Figure(go.Heatmap(
    z=program_heatmap.to_numpy(), x=program_heatmap.columns, y=program_heatmap.index,
    colorscale=[[0, '#f8fafc'], [0.35, '#93c5fd'], [1, '#1d4ed8']],
    colorbar=dict(title='% del programa', tickformat='.0%'),
    hovertemplate='<b>%{y}</b><br>%{x}<br>%{z:.1%}<extra></extra>',
))
fig_program_cluster_heatmap.update_layout(
    title='Perfil temático de cada programa', width=1180, height=760,
    margin=dict(l=270, r=30, t=70, b=200), xaxis_tickangle=35,
)
fig_program_cluster_heatmap.show()

fig_program_similarity = go.Figure(go.Heatmap(
    z=program_similarity_matrix, x=program_order, y=program_order,
    zmin=0, zmax=1,
    colorscale=[[0, '#f8fafc'], [0.5, '#fbbf24'], [1, '#be123c']],
    colorbar=dict(title='Similitud'),
    hovertemplate='<b>%{y}</b><br>%{x}<br>coseno=%{z:.3f}<extra></extra>',
))
fig_program_similarity.update_layout(
    title='Similitud entre programas según su mezcla de temas', width=1080, height=880,
    margin=dict(l=270, r=30, t=70, b=260), xaxis_tickangle=45,
)
fig_program_similarity.show()

interdisciplinary_plot = cluster_interdisciplinarity.sort_values('entropia_programas').copy()
fig_interdisciplinarity = px.bar(
    interdisciplinary_plot,
    x='entropia_programas', y='cluster_theme', color='nivel_dominante', orientation='h',
    color_discrete_map=LEVEL_COLOR_MAP,
    hover_data=['conteo', 'n_programas', 'programas_efectivos', 'programa_dominante', 'programas_top'],
    title='Temas más interdisciplinarios', width=980, height=650,
    labels={'entropia_programas': 'Entropía normalizada entre programas', 'cluster_theme': 'Tema', 'nivel_dominante': 'Nivel dominante'},
)
fig_interdisciplinarity.update_layout(margin=dict(l=220, r=25, t=70, b=50), legend_title_text='Nivel dominante')
fig_interdisciplinarity.show()

fig_level_time = px.line(
    level_year, x='anio_pub', y='conteo', color='nivel', markers=True,
    category_orders={'nivel': level_order}, color_discrete_map=LEVEL_COLOR_MAP,
    title='Producción de tesis por nivel a través del tiempo', width=1040, height=500,
    labels={'anio_pub': 'Año', 'conteo': 'Tesis', 'nivel': 'Nivel'},
)
fig_level_time.update_layout(margin=dict(l=55, r=25, t=70, b=50), legend_title_text='Nivel')
fig_level_time.show()

display(cluster_interdisciplinarity.head(12))
display(program_cluster_summary)
display(reliable_program_similarity.head(15))


## Cruce asesores × temas

Ahora que los asesores están homologados, se puede medir concentración temática y diversidad por asesor.


In [ ]:
advisor_long = df_layout.copy()
if 'tesis_row_id' not in advisor_long.columns:
    advisor_long.insert(0, 'tesis_row_id', advisor_long.index.astype(int))
advisor_long = (
    advisor_long
    .assign(asesor_unificado=lambda d: d['asesor_unificado'].fillna('Asesor desconocido').str.split(r'\s*\|\s*'))
    .explode('asesor_unificado')
)
advisor_long['asesor_unificado'] = advisor_long['asesor_unificado'].str.strip()
advisor_long = advisor_long[
    advisor_long['asesor_unificado'].notna()
    & advisor_long['asesor_unificado'].ne('')
    & advisor_long['asesor_unificado'].ne('Asesor desconocido')
].copy()

advisor_cluster = (
    advisor_long.groupby(['asesor_unificado', 'cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label'])
    .agg(
        n_tesis=('tesis_row_id', 'nunique'),
        n_programas=('grado_programa', 'nunique'),
        anio_min=('anio_pub', 'min'),
        anio_max=('anio_pub', 'max'),
    )
    .reset_index()
)
advisor_cluster['total_asesor'] = advisor_cluster.groupby('asesor_unificado')['n_tesis'].transform('sum')
advisor_cluster['participacion_asesor'] = advisor_cluster['n_tesis'] / advisor_cluster['total_asesor']
advisor_cluster = advisor_cluster.sort_values(['total_asesor', 'asesor_unificado', 'n_tesis'], ascending=[False, True, False])
advisor_cluster.to_parquet(ADVISOR_CLUSTER_PATH, index=False)

advisor_program_count = advisor_long.groupby('asesor_unificado')['grado_programa'].nunique()
advisor_summary = (
    advisor_cluster.sort_values(['asesor_unificado', 'n_tesis'], ascending=[True, False])
    .groupby('asesor_unificado')
    .agg(
        n_tesis=('n_tesis', 'sum'),
        n_clusters=('cluster_id', 'nunique'),
        cluster_principal=('cluster_theme', 'first'),
        cluster_principal_id=('cluster_label', 'first'),
        participacion_cluster_principal=('participacion_asesor', 'first'),
        anio_min=('anio_min', 'min'),
        anio_max=('anio_max', 'max'),
    )
    .reset_index()
)
advisor_summary['n_programas'] = advisor_summary['asesor_unificado'].map(advisor_program_count).fillna(0).astype(int)
advisor_summary = advisor_summary.sort_values('n_tesis', ascending=False)
advisor_summary.to_parquet(ADVISOR_SUMMARY_PATH, index=False)

display(advisor_summary.head(25))

fig_advisor = px.scatter(
    advisor_summary.head(45),
    x='n_tesis', y='n_clusters', size='n_programas', color='participacion_cluster_principal',
    hover_name='asesor_unificado',
    hover_data=['n_programas', 'cluster_principal', 'cluster_principal_id', 'anio_min', 'anio_max'],
    title='Volumen, diversidad temática y alcance entre programas del profesorado',
    color_continuous_scale='Tealrose', width=980, height=580,
    labels={
        'n_tesis': 'Tesis asesoradas',
        'n_clusters': 'Temas distintos',
        'n_programas': 'Programas',
        'participacion_cluster_principal': 'Concentración en tema principal',
    },
)
fig_advisor.update_traces(marker=dict(line=dict(width=0.6, color='white'), opacity=0.86))
fig_advisor.update_layout(margin=dict(l=60, r=30, t=70, b=50))
fig_advisor.show()


## Top asesores por cluster


In [ ]:
top_advisor_figs = []
for cid in sorted(df_layout['cluster_id'].unique()):
    cluster_label = f'Cluster {cid:02d}'
    cluster_theme = cluster_theme_by_id.get(cid, cluster_label)
    subset = advisor_long[advisor_long['cluster_id'] == cid]
    top5 = (
        subset.groupby('asesor_unificado')
        .size()
        .reset_index(name='conteo')
        .sort_values('conteo', ascending=False)
        .head(5)
    )

    if top5.empty:
        continue

    fig_top_advisors = px.bar(
        top5, x='conteo', y='asesor_unificado', orientation='h',
        title=f'Top 5 asesores: {cluster_theme}', text='conteo',
        color_discrete_sequence=[cluster_color_map.get(cluster_label, '#2563eb')],
    )
    fig_top_advisors.update_traces(textposition='outside', marker_line_width=0)
    fig_top_advisors.update_layout(
        height=320, xaxis_title='Número de tesis', yaxis_title='Asesor/a',
        yaxis=dict(categoryorder='array', categoryarray=top5['asesor_unificado'].tolist()[::-1]),
        showlegend=False, margin=dict(t=60, b=40, l=40, r=20),
    )
    top_advisor_figs.append(fig_top_advisors)
    fig_top_advisors.show()


## Exportar resultados analíticos


In [ ]:
cols_export = [
    'cluster_label', 'cluster_id', 'cluster_theme', 'cluster_display_label',
    'titulo', 'resumen', 'materias', 'autorx', 'anio_pub', 'periodo',
    'asesor_unificado', 'idioma', 'nivel', 'programa', 'grado_programa',
    'coleccion', 'collection_handle', 'item_handle', 'url',
]

extra_cluster_export_cols = [
    'cluster_id_consensus_ensemble', 'cluster_label_consensus_ensemble', 'cluster_theme_consensus_ensemble',
    'consensus_primary_theme', 'consensus_secondary_theme', 'consensus_membership_margin',
    'macro_cluster_id', 'macro_cluster_theme', 'taxonomy_label',
]
extra_cluster_export_cols = [col for col in extra_cluster_export_cols if col in df_layout.columns]

clusters_export = (
    df_layout[cols_export + extra_cluster_export_cols + ['umap_x', 'umap_y', 'umap_z']]
    .sort_values(['cluster_id', 'anio_pub', 'titulo'])
    .reset_index(drop=True)
)

variant_cols = []
for variant in cluster_solutions:
    variant_cols.extend([
        f'cluster_id_{variant}',
        f'cluster_label_{variant}',
        f'cluster_theme_{variant}',
        f'cluster_display_label_{variant}',
    ])

cluster_variant_assignments = (
    df_layout[['titulo', 'autorx', 'anio_pub', 'periodo', 'asesor_unificado', 'idioma',
               'nivel', 'programa', 'grado_programa', 'coleccion', 'collection_handle', 'item_handle',
               'url', 'umap_x', 'umap_y', 'umap_z'] + variant_cols]
    .sort_values(['anio_pub', 'titulo'])
    .reset_index(drop=True)
)

clusters_export.to_parquet(CLUSTERS_PATH, index=False)
cluster_overview.to_parquet(CLUSTER_SUMMARY_PATH, index=False)
cluster_variant_assignments.to_parquet(CLUSTER_VARIANTS_PATH, index=False)
cluster_variant_summary.to_parquet(CLUSTER_VARIANT_SUMMARY_PATH, index=False)
cluster_variant_metrics.to_parquet(CLUSTER_VARIANT_METRICS_PATH, index=False)
topic_model_metrics.to_parquet(TOPIC_MODEL_METRICS_PATH, index=False)
keyword_overlap_pairs.to_parquet(CLUSTER_KEYWORD_OVERLAP_PATH, index=False)
topic_model_frontier.to_parquet(TOPIC_MODEL_FRONTIER_PATH, index=False)
topic_stability.to_parquet(TOPIC_STABILITY_PATH, index=False)
topic_memberships.to_parquet(TOPIC_MEMBERSHIP_PATH, index=False)
topic_taxonomy.to_parquet(TOPIC_TAXONOMY_PATH, index=False)
cluster_program.to_parquet(CLUSTER_PROGRAM_PATH, index=False)
cluster_level.to_parquet(CLUSTER_LEVEL_PATH, index=False)
cluster_interdisciplinarity.to_parquet(CLUSTER_INTERDISCIPLINARITY_PATH, index=False)
program_cluster_summary.to_parquet(PROGRAM_CLUSTER_SUMMARY_PATH, index=False)
program_similarity.to_parquet(PROGRAM_SIMILARITY_PATH, index=False)
program_year.to_parquet(PROGRAM_YEAR_PATH, index=False)

print(f'Archivo exportado: {CLUSTERS_PATH.resolve()}')
print(f'Resumen exportado: {CLUSTER_SUMMARY_PATH.resolve()}')
print(f'Asignaciones por variante exportadas: {CLUSTER_VARIANTS_PATH.resolve()}')
print(f'Resumen por variante exportado: {CLUSTER_VARIANT_SUMMARY_PATH.resolve()}')
print(f'Métricas por variante exportadas: {CLUSTER_VARIANT_METRICS_PATH.resolve()}')
print(f'Métricas de topic models exportadas: {TOPIC_MODEL_METRICS_PATH.resolve()}')
print(f'Traslape de keywords exportado: {CLUSTER_KEYWORD_OVERLAP_PATH.resolve()}')
print(f'Frontera Pareto de topic models exportada: {TOPIC_MODEL_FRONTIER_PATH.resolve()}')
print(f'Estabilidad de topic models exportada: {TOPIC_STABILITY_PATH.resolve()}')
print(f'Membresias suaves de consenso exportadas: {TOPIC_MEMBERSHIP_PATH.resolve()}')
print(f'Taxonomia jerarquica exportada: {TOPIC_TAXONOMY_PATH.resolve()}')
print(f'Composicion cluster-programa exportada: {CLUSTER_PROGRAM_PATH.resolve()}')
print(f'Composicion cluster-nivel exportada: {CLUSTER_LEVEL_PATH.resolve()}')
print(f'Interdisciplinariedad de clusters exportada: {CLUSTER_INTERDISCIPLINARITY_PATH.resolve()}')
print(f'Perfiles tematicos de programas exportados: {PROGRAM_CLUSTER_SUMMARY_PATH.resolve()}')
print(f'Similitud entre programas exportada: {PROGRAM_SIMILARITY_PATH.resolve()}')
print(f'Evolucion por programa exportada: {PROGRAM_YEAR_PATH.resolve()}')
print(f'Diagnóstico exportado: {CLUSTER_DIAGNOSTICS_PATH.resolve()}')
print(f'Diagnóstico UMAP exportado: {UMAP_DIAGNOSTICS_PATH.resolve()}')
print(f'Evolución temporal exportada: {CLUSTER_YEAR_PATH.resolve()}')
print(f'Distribución por idioma exportada: {CLUSTER_LANGUAGE_PATH.resolve()}')
print(f'Cruce asesor-cluster exportado: {ADVISOR_CLUSTER_PATH.resolve()}')
print(f'Resumen de asesores exportado: {ADVISOR_SUMMARY_PATH.resolve()}')


## Red de tesis por asesor

Esta red conecta tesis asesoradas por la misma persona. El layout agrupa primero por cluster semántico para que las comunidades temáticas queden juntas, y mantiene las aristas de asesoría como puentes institucionales entre temas.


In [ ]:
graph_df = df_layout.copy()

G = nx.Graph()
G.add_nodes_from(graph_df.index)

for advisor, ids_series in advisor_long.groupby('asesor_unificado')['tesis_row_id']:
    ids = sorted(set(ids_series.astype(int).tolist()))
    if len(ids) < 2:
        continue
    for i, j in combinations(ids, 2):
        w = G.get_edge_data(i, j, {}).get('weight', 0) + 1
        G.add_edge(i, j, weight=w, advisor=advisor)

if G.number_of_edges() == 0:
    print('No hay conexiones por asesor (todos únicos o faltantes).')
    df_layout['advisor_comm_id'] = -1
    df_layout['advisor_comm_label'] = 'Sin comunidad'
else:
    communities = list(nx.algorithms.community.greedy_modularity_communities(G, weight='weight'))
    comm_map = {node: cid for cid, nodes in enumerate(communities) for node in nodes}
    df_layout['advisor_comm_id'] = df_layout.index.map(lambda x: comm_map.get(x, -1))
    df_layout['advisor_comm_label'] = df_layout['advisor_comm_id'].map(lambda x: f'AdvisorComm {x}' if x != -1 else 'Sin comunidad')

    comm_summary = (
        df_layout.groupby('advisor_comm_label')
        .agg(
            conteo=('titulo', 'count'),
            asesores_top=('asesor_unificado', lambda s: ', '.join(s.value_counts().head(3).index)),
        )
        .sort_values('conteo', ascending=False)
    )
    print(f'Comunidades detectadas: {len(communities)}')
    display(comm_summary.head(20))


## Visualizar red de asesores agrupada por cluster semántico


In [ ]:
def hex_to_rgba(hex_color, alpha=0.12):
    hex_color = hex_color.lstrip('#')
    r, g, b = (int(hex_color[i:i + 2], 16) for i in (0, 2, 4))
    return f'rgba({r},{g},{b},{alpha})'

def convex_hull(points):
    pts = sorted(set(map(tuple, points)))
    if len(pts) <= 2:
        return pts

    def cross(o, a, b):
        return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])

    lower = []
    for p in pts:
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) <= 0:
            lower.pop()
        lower.append(p)

    upper = []
    for p in reversed(pts):
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) <= 0:
            upper.pop()
        upper.append(p)

    return lower[:-1] + upper[:-1]

if 'advisor_comm_label' not in df_layout.columns or G.number_of_edges() == 0:
    fig_network = None
    print('No hay red de asesores para visualizar.')
else:
    cluster_counts = df_layout['cluster_label'].value_counts().reindex(cluster_order)
    n_clusters = len(cluster_order)
    n_cols = int(np.ceil(np.sqrt(n_clusters)))
    n_rows = int(np.ceil(n_clusters / n_cols))
    anchor_spacing_x = 4.7
    anchor_spacing_y = 3.8

    cluster_anchors = {}
    for order_idx, cluster_name in enumerate(cluster_order):
        row, col = divmod(order_idx, n_cols)
        x = (col - (n_cols - 1) / 2) * anchor_spacing_x
        y = ((n_rows - 1) / 2 - row) * anchor_spacing_y
        cluster_anchors[cluster_name] = np.array([x, y], dtype=float)

    pos = {}
    cluster_radii = {}
    for cid, cluster_name in enumerate(cluster_order):
        node_ids = df_layout.index[df_layout['cluster_label'].eq(cluster_name)].tolist()
        subgraph = G.subgraph(node_ids)
        radius = 0.72 + 0.06 * np.sqrt(len(node_ids))
        cluster_radii[cluster_name] = radius

        if len(node_ids) == 1:
            local_pos = {node_ids[0]: np.array([0.0, 0.0])}
        else:
            local_pos = nx.spring_layout(
                subgraph,
                weight='weight',
                seed=RANDOM_STATE + cid,
                k=0.9 / np.sqrt(max(len(node_ids), 2)),
                iterations=180,
            )
            coords = np.array([local_pos[node] for node in node_ids], dtype=float)
            coords = coords - coords.mean(axis=0)
            max_abs = np.abs(coords).max()
            if max_abs > 0:
                coords = coords / max_abs
            for node, coord in zip(node_ids, coords):
                local_pos[node] = coord

        anchor = cluster_anchors[cluster_name]
        for node in node_ids:
            degree_weight = G.degree(node, weight='weight') or 0
            pull_to_center = 1 - min(0.18, degree_weight * 0.012)
            pos[node] = anchor + np.asarray(local_pos[node]) * radius * pull_to_center

    nodes_df = (
        df_layout[['titulo', 'anio_pub', 'idioma', 'asesor_unificado', 'cluster_label', 'cluster_theme', 'cluster_display_label', 'advisor_comm_label']]
        .assign(
            x=lambda d: d.index.map(lambda i: pos[i][0]),
            y=lambda d: d.index.map(lambda i: pos[i][1]),
            degree=lambda d: d.index.map(lambda i: G.degree(i, weight='weight')),
        )
    )
    nodes_df['node_size'] = np.clip(6 + np.sqrt(nodes_df['degree'].fillna(0)) * 2.2, 6, 16)
    nodes_df['color'] = nodes_df['cluster_label'].map(cluster_color_map)

    hull_traces = []
    annotations = []
    for cluster_name in cluster_order:
        points = nodes_df.loc[nodes_df['cluster_label'].eq(cluster_name), ['x', 'y']].to_numpy()
        color = cluster_color_map.get(cluster_name, '#2563eb')
        anchor = cluster_anchors[cluster_name]

        if len(points) >= 3:
            hull = np.array(convex_hull(points))
            center = hull.mean(axis=0)
            hull = center + (hull - center) * 1.18
            hull = np.vstack([hull, hull[0]])
            hull_traces.append(
                go.Scatter(
                    x=hull[:, 0],
                    y=hull[:, 1],
                    mode='lines',
                    line=dict(color=hex_to_rgba(color, 0.42), width=1.1),
                    fill='toself',
                    fillcolor=hex_to_rgba(color, 0.10),
                    hoverinfo='skip',
                    showlegend=False,
                )
            )

        annotations.append(
            dict(
                x=anchor[0],
                y=anchor[1] + cluster_radii[cluster_name] + 0.45,
                text=f'{cluster_label_to_theme.get(cluster_name, cluster_name)}<br><span style="font-size:11px;color:#64748b">{cluster_name} · {int(cluster_counts.get(cluster_name, 0))} tesis</span>',
                showarrow=False,
                font=dict(size=12, color='#111827'),
                align='center',
                bgcolor='rgba(255,255,255,0.82)',
                bordercolor='rgba(148,163,184,0.45)',
                borderwidth=1,
                borderpad=4,
            )
        )

    intra_edge_x, intra_edge_y, cross_edge_x, cross_edge_y = [], [], [], []
    for u, v in G.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        same_cluster = df_layout.loc[u, 'cluster_label'] == df_layout.loc[v, 'cluster_label']
        target_x = intra_edge_x if same_cluster else cross_edge_x
        target_y = intra_edge_y if same_cluster else cross_edge_y
        target_x += [x0, x1, None]
        target_y += [y0, y1, None]

    cross_edge_trace = go.Scatter(
        x=cross_edge_x,
        y=cross_edge_y,
        line=dict(width=0.55, color='rgba(15,23,42,0.16)', dash='dot'),
        hoverinfo='none',
        mode='lines',
        showlegend=False,
    )
    intra_edge_trace = go.Scatter(
        x=intra_edge_x,
        y=intra_edge_y,
        line=dict(width=0.50, color='rgba(71,85,105,0.18)'),
        hoverinfo='none',
        mode='lines',
        showlegend=False,
    )

    node_traces = []
    for cluster_name in cluster_order:
        subset = nodes_df[nodes_df['cluster_label'].eq(cluster_name)]
        if subset.empty:
            continue
        node_traces.append(
            go.Scatter(
                x=subset['x'],
                y=subset['y'],
                mode='markers',
                name=cluster_label_to_theme.get(cluster_name, cluster_name),
                marker=dict(
                    size=subset['node_size'],
                    color=cluster_color_map.get(cluster_name, '#2563eb'),
                    opacity=0.88,
                    line=dict(width=0.7, color='white'),
                ),
                text=subset.apply(
                    lambda r: (
                        f"{html.escape(str(r['titulo']))}<br>"
                        f"Asesor: {html.escape(str(r['asesor_unificado']))}<br>"
                        f"Tema: {html.escape(str(r['cluster_theme']))}<br>"
                        f"Cluster técnico: {html.escape(str(r['cluster_label']))}<br>"
                        f"Comunidad asesoría: {html.escape(str(r['advisor_comm_label']))}<br>"
                        f"Año: {html.escape(str(r['anio_pub']))}<br>"
                        f"Idioma: {html.escape(str(r['idioma']))}"
                    ),
                    axis=1,
                ),
                hoverinfo='text',
            )
        )

    fig_network = go.Figure(data=hull_traces + [cross_edge_trace, intra_edge_trace] + node_traces)
    fig_network.update_layout(
        title='Red de tesis por asesor agrupada por tema semántico',
        legend_title_text='Tema semántico',
        width=1080,
        height=780,
        margin=dict(l=25, r=25, t=72, b=25),
        plot_bgcolor='white',
        paper_bgcolor='white',
        annotations=annotations,
        xaxis=dict(visible=False, scaleanchor='y', scaleratio=1),
        yaxis=dict(visible=False),
    )
    fig_network.show()
    print('Layout de red: nodos agrupados por tema semántico; líneas sólidas dentro de tema y punteadas entre temas.')


## Dashboard HTML

Se exporta una vista interactiva en HTML con mapa temporal, diagnóstico de clusters, evolución temática, mezcla de idioma y cruces de asesoría.

In [ ]:

def fig_html(fig, div_id, include_js=False):
    if fig is None:
        return '<div class="empty-state">Sin datos suficientes para esta visualización.</div>'
    return pio.to_html(fig, include_plotlyjs=include_js, full_html=False, div_id=div_id)

def table_html(frame, columns, max_rows=12, formatters=None):
    present = [c for c in columns if c in frame.columns]
    if frame.empty or not present:
        return '<div class="empty-state">Sin filas para mostrar.</div>'
    view = frame[present].head(max_rows).copy()
    if formatters:
        for col, formatter in formatters.items():
            if col in view.columns:
                view[col] = view[col].map(formatter)
    return view.to_html(index=False, classes='data-table', border=0, escape=True)

selected_diag = diagnostics.query('feature_space == "embeddings" and k == @selected_k').iloc[0]
best_diag = diagnostics.query('feature_space == "embeddings"').sort_values(['silhouette', 'davies_bouldin'], ascending=[False, True]).iloc[0]
english_clusters = 0
if not cluster_language.empty and (cluster_language['idioma'] == 'eng').any():
    english_clusters = int(cluster_language.loc[cluster_language['idioma'] == 'eng', 'cluster_label'].nunique())

year_min = int(df_layout['anio_pub'].min())
year_max = int(df_layout['anio_pub'].max())
latest_year = int(df_layout['anio_pub'].max())
latest_total = int((df_layout['anio_pub'].astype('Int64') == latest_year).sum())
fastest_growth = cluster_lifecycle.sort_values('cambio_2020s_vs_2010s_pp', ascending=False).iloc[0]
best_keyword_variant = cluster_variant_metrics.sort_values(['mean_keyword_jaccard', 'max_keyword_jaccard']).iloc[0]
best_coherence_variant = cluster_variant_metrics.sort_values('topic_coherence_cv', ascending=False).iloc[0]
best_interpretability_variant = cluster_variant_metrics.sort_values('interpretability_score', ascending=False).iloc[0]
bertopic_metric = cluster_variant_metrics.loc[cluster_variant_metrics['variant'].eq('bertopic_hdbscan')].iloc[0]
best_robust_variant = cluster_variant_metrics.sort_values('robust_interpretability_score', ascending=False).iloc[0]
frontier_count = int(cluster_variant_metrics['pareto_frontier'].sum())
consensus_metric = cluster_variant_metrics.loc[cluster_variant_metrics['variant'].eq('consensus_ensemble')].iloc[0]
ambiguous_share = float((df_layout['consensus_membership_margin'] < df_layout['consensus_membership_margin'].quantile(0.25)).mean())
most_interdisciplinary = cluster_interdisciplinarity.iloc[0]
widest_program = program_cluster_summary.sort_values('entropia_tematica', ascending=False).iloc[0]
closest_program_pair = reliable_program_similarity.iloc[0] if not reliable_program_similarity.empty else program_similarity_pairs.iloc[0]

metric_cards = [
    ('Tesis analizadas', f'{len(df_layout):,}'),
    ('Años cubiertos', f'{year_min}-{year_max}'),
    ('Niveles académicos', f"{df_layout['nivel'].nunique()}"),
    ('Programas por nivel', f"{df_layout['grado_programa'].nunique()}"),
    ('Clusters activos', f'{selected_k}'),
    ('Silhouette k usado', f'{selected_diag["silhouette"]:.3f}'),
    ('Trustworthiness 2D', f'{umap_trust_2d:.3f}'),
    ('Trustworthiness 3D', f'{umap_trust_3d:.3f}'),
    ('Menor traslape keywords', f"{best_keyword_variant['variant_label'].replace('KMeans sobre ', '')}"),
    ('Mejor score interpretativo', f"{best_interpretability_variant['variant_label']} ({best_interpretability_variant['interpretability_score']:.3f})"),
    ('Mejor score robusto', f"{best_robust_variant['variant_label']} ({best_robust_variant['robust_interpretability_score']:.3f})"),
    ('Modelos en frontera Pareto', f'{frontier_count}'),
    ('Estabilidad consenso', f"{consensus_metric['stability_score']:.3f}"),
    ('Mejor coherencia c_v', f"{best_coherence_variant['variant_label']} ({best_coherence_variant['topic_coherence_cv']:.3f})"),
    ('Tópicos BERTopic', f"{int(bertopic_metric['n_topics'])}"),
    ('Outliers BERTopic', f"{bertopic_metric['outlier_rate']:.1%}"),
    ('Clusters con tesis en inglés', f'{english_clusters}'),
    ('Tema más interdisciplinario', f"{most_interdisciplinary['cluster_theme']} ({most_interdisciplinary['entropia_programas']:.2f})"),
    ('Programa más diverso', f"{widest_program['grado_programa']} ({widest_program['entropia_tematica']:.2f})"),
]
metric_html = ''.join(f'<section class="metric"><span>{html.escape(label)}</span><strong>{html.escape(value)}</strong></section>' for label, value in metric_cards)

cluster_table = table_html(
    cluster_overview.sort_values('cluster_id'),
    ['cluster_label', 'cluster_theme', 'conteo', 'n_programas', 'programas_top', 'anio_promedio', 'idiomas', 'keywords', 'asesores_top'],
    selected_k,
)
program_summary_table = table_html(
    program_cluster_summary.sort_values(['nivel', 'n_tesis'], ascending=[True, False]),
    ['nivel', 'programa', 'n_tesis', 'n_clusters', 'entropia_tematica', 'cluster_principal', 'participacion_cluster_principal', 'temas_top'],
    len(program_cluster_summary),
    formatters={
        'entropia_tematica': lambda x: f'{x:.3f}',
        'participacion_cluster_principal': lambda x: f'{x:.1%}',
    },
)
interdisciplinarity_table = table_html(
    cluster_interdisciplinarity,
    ['cluster_label', 'cluster_theme', 'conteo', 'n_programas', 'n_niveles', 'entropia_programas', 'programas_efectivos', 'programa_dominante', 'participacion_programa_dominante', 'programas_top'],
    selected_k,
    formatters={
        'entropia_programas': lambda x: f'{x:.3f}',
        'programas_efectivos': lambda x: f'{x:.1f}',
        'participacion_programa_dominante': lambda x: f'{x:.1%}',
    },
)
program_similarity_table = table_html(
    reliable_program_similarity,
    ['programa_a', 'programa_b', 'similitud_coseno', 'n_tesis_a', 'n_tesis_b'],
    15,
    formatters={'similitud_coseno': lambda x: f'{x:.3f}'},
)
lifecycle_table = table_html(
    cluster_lifecycle.sort_values('cluster_id'),
    ['cluster_label', 'cluster_theme', 'conteo', 'anio_inicio', 'anio_pico', 'conteo_pico', 'participacion_pico', 'cambio_2020s_vs_2010s_pp', 'keywords'],
    selected_k,
    formatters={
        'participacion_pico': lambda x: f'{x:.1f}%',
        'cambio_2020s_vs_2010s_pp': lambda x: f'{x:+.1f} pp',
    },
)
advisor_table = table_html(advisor_summary, ['asesor_unificado', 'n_tesis', 'n_programas', 'n_clusters', 'cluster_principal', 'cluster_principal_id', 'participacion_cluster_principal', 'anio_min', 'anio_max'], 20)
benchmark_table = table_html(benchmark_view if 'benchmark_view' in globals() else pd.DataFrame(), ['model_name', 'embedding_dim', 'seconds', 'k', 'silhouette_cosine', 'language_nmi', 'umap_trustworthiness'], 10)
umap_quality_table = table_html(umap_diagnostics, ['projection', 'n_components', 'trustworthiness', 'trustworthiness_delta_vs_2d', 'interpretation'], 2, formatters={'trustworthiness': lambda x: f'{x:.3f}', 'trustworthiness_delta_vs_2d': lambda x: f'{x:+.3f}'})
variant_metrics_table = table_html(
    cluster_variant_metrics.sort_values(['mean_keyword_jaccard', 'max_keyword_jaccard']),
    ['variant_label', 'algorithm', 'n_topics', 'outlier_rate', 'topic_coherence_cv', 'topic_diversity', 'mean_keyword_jaccard', 'max_keyword_jaccard', 'largest_topic_share', 'singleton_topics', 'interpretability_score', 'silhouette', 'language_nmi', 'program_nmi', 'level_nmi', 'min_cluster_size', 'max_cluster_size', 'reused_keyword_examples'],
    len(cluster_variant_metrics),
    formatters={
        'outlier_rate': lambda x: f'{x:.1%}',
        'silhouette': lambda x: f'{x:.3f}',
        'language_nmi': lambda x: f'{x:.3f}',
        'program_nmi': lambda x: f'{x:.3f}',
        'level_nmi': lambda x: f'{x:.3f}',
        'topic_coherence_cv': lambda x: f'{x:.3f}',
        'topic_coherence_npmi': lambda x: f'{x:.3f}',
        'topic_diversity': lambda x: f'{x:.3f}',
        'mean_keyword_jaccard': lambda x: f'{x:.3f}',
        'max_keyword_jaccard': lambda x: f'{x:.3f}',
        'largest_topic_share': lambda x: f'{x:.1%}',
        'interpretability_score': lambda x: f'{x:.3f}',
        'robust_interpretability_score': lambda x: f'{x:.3f}',
        'stability_score': lambda x: f'{x:.3f}',
    },
)
frontier_table = table_html(
    topic_model_frontier.sort_values('robust_interpretability_score', ascending=False),
    ['variant_label', 'algorithm', 'n_topics', 'topic_coherence_cv', 'topic_coherence_npmi', 'mean_keyword_jaccard', 'largest_topic_share', 'stability_score', 'robust_interpretability_score'],
    len(topic_model_frontier),
    formatters={
        'topic_coherence_cv': lambda x: f'{x:.3f}',
        'topic_coherence_npmi': lambda x: f'{x:.3f}',
        'mean_keyword_jaccard': lambda x: f'{x:.3f}',
        'largest_topic_share': lambda x: f'{x:.1%}',
        'stability_score': lambda x: f'{x:.3f}',
        'robust_interpretability_score': lambda x: f'{x:.3f}',
    },
)
stability_table = table_html(
    topic_stability.merge(cluster_variant_metrics[['variant', 'variant_label', 'pareto_frontier']], on='variant', how='left').sort_values('stability_score', ascending=False),
    ['variant_label', 'pareto_frontier', 'seed_stability_nmi', 'bootstrap_stability_nmi', 'assignment_margin_mean', 'stability_score'],
    len(topic_stability),
    formatters={
        'seed_stability_nmi': lambda x: f'{x:.3f}' if pd.notna(x) else '',
        'bootstrap_stability_nmi': lambda x: f'{x:.3f}' if pd.notna(x) else '',
        'assignment_margin_mean': lambda x: f'{x:.3f}' if pd.notna(x) else '',
        'stability_score': lambda x: f'{x:.3f}' if pd.notna(x) else '',
    },
)
taxonomy_table = table_html(
    topic_taxonomy.sort_values(['macro_cluster_id', 'conteo'], ascending=[True, False]),
    ['macro_theme', 'subtopic_label', 'subtopic_theme', 'conteo', 'macro_share', 'keywords'],
    22,
    formatters={'macro_share': lambda x: f'{x:.1%}'},
)
ambiguous_memberships = (
    topic_memberships[topic_memberships['rank'].eq(1)]
    .merge(df_layout[['tesis_row_id', 'consensus_membership_margin', 'consensus_secondary_theme']], on='tesis_row_id', how='left')
    .sort_values('consensus_membership_margin')
)
membership_table = table_html(
    ambiguous_memberships,
    ['titulo', 'anio_pub', 'consensus_cluster_theme', 'consensus_secondary_theme', 'membership_score', 'membership_share', 'consensus_membership_margin'],
    12,
    formatters={
        'membership_score': lambda x: f'{x:.3f}',
        'membership_share': lambda x: f'{x:.1%}',
        'consensus_membership_margin': lambda x: f'{x:.3f}',
    },
)
keyword_overlap_table = table_html(
    keyword_overlap_pairs.sort_values(['keyword_jaccard', 'n_shared_keywords'], ascending=[False, False]),
    ['variant_label', 'cluster_a', 'cluster_b', 'cluster_a_theme', 'cluster_b_theme', 'n_shared_keywords', 'keyword_jaccard', 'shared_keywords'],
    12,
    formatters={'keyword_jaccard': lambda x: f'{x:.3f}'},
)

time_machine_html = fig_html(fig_time_machine, 'time-machine', include_js=True)
periods_html = fig_html(fig_period_facets, 'period-facets')
map_html = fig_html(fig_map, 'semantic-map')
map_3d_html = fig_html(fig_map_3d, 'semantic-map-3d')
umap_quality_html = fig_html(fig_umap_quality, 'umap-quality')
variant_map_2d_html = fig_html(fig_map_variant_2d, 'variant-map-2d')
variant_map_3d_html = fig_html(fig_map_variant_3d, 'variant-map-3d')
bertopic_map_html = fig_html(fig_map_bertopic, 'bertopic-map')
variant_overlap_html = fig_html(fig_variant_overlap, 'variant-keyword-overlap')
topic_quality_html = fig_html(fig_topic_quality, 'topic-quality')
consensus_map_html = fig_html(fig_map_consensus, 'consensus-map')
frontier_html = fig_html(fig_topic_frontier, 'topic-frontier')
stability_html = fig_html(fig_topic_stability, 'topic-stability')
taxonomy_html = fig_html(fig_topic_taxonomy, 'topic-taxonomy')
membership_html = fig_html(fig_membership_ambiguity, 'membership-ambiguity')
diag_html = fig_html(fig_diag, 'cluster-diagnostics')
year_html = fig_html(fig_year_heatmap, 'year-heatmap')
stream_html = fig_html(fig_theme_stream, 'theme-stream')
race_html = fig_html(fig_theme_race, 'theme-race')
language_html = fig_html(fig_language, 'language-distribution')
advisor_html = fig_html(fig_advisor, 'advisor-scatter')
network_html = fig_html(fig_network, 'advisor-network')
map_level_html = fig_html(fig_map_level, 'map-level')
cluster_level_html = fig_html(fig_cluster_level, 'cluster-level')
program_cluster_html = fig_html(fig_program_cluster_heatmap, 'program-cluster')
program_similarity_html = fig_html(fig_program_similarity, 'program-similarity')
interdisciplinarity_html = fig_html(fig_interdisciplinarity, 'cluster-interdisciplinarity')
level_time_html = fig_html(fig_level_time, 'level-time')

html_doc = f'''<!doctype html>
<html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Mapa semántico de tesis CIDE</title>
<style>
:root {{ --ink:#111827; --muted:#64748b; --line:#d8dee8; --panel:#fff; --soft:#f6f8fb; --accent:#2563eb; --accent2:#d97706; }}
* {{ box-sizing:border-box; }}
html {{ scroll-behavior:smooth; }}
body {{ margin:0; font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif; color:var(--ink); background:#f8fafc; }}
header {{ padding:28px 32px 18px; border-bottom:1px solid var(--line); background:#fff; }}
h1 {{ margin:0 0 8px; font-size:30px; line-height:1.15; letter-spacing:0; }}
header p {{ margin:0; color:var(--muted); max-width:1120px; line-height:1.45; }}
nav {{ display:flex; gap:8px; flex-wrap:wrap; margin-top:16px; }}
nav a {{ color:#1f2937; text-decoration:none; border:1px solid var(--line); background:#fff; border-radius:8px; padding:7px 10px; font-size:13px; }}
main {{ padding:24px 32px 40px; }}
.metrics {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(170px,1fr)); gap:12px; margin-bottom:18px; }}
.metric {{ background:var(--panel); border:1px solid var(--line); border-radius:8px; padding:13px 14px; }}
.metric span {{ display:block; color:var(--muted); font-size:12px; }}
.metric strong {{ display:block; margin-top:4px; font-size:24px; }}
.grid {{ display:grid; grid-template-columns:minmax(0,1fr); gap:18px; }}
.panel {{ background:var(--panel); border:1px solid var(--line); border-radius:8px; padding:16px; overflow:hidden; }}
.panel h2 {{ margin:0 0 10px; font-size:18px; }}
.panel p {{ color:var(--muted); line-height:1.45; }}
.two-col {{ display:grid; grid-template-columns:minmax(0,1fr) minmax(0,1fr); gap:18px; }}
.callout {{ border-left:4px solid var(--accent); padding:8px 12px; background:#f8fafc; margin:0 0 12px; color:#243043; }}
.data-table {{ width:100%; border-collapse:collapse; font-size:13px; }}
.data-table th,.data-table td {{ border-bottom:1px solid var(--line); padding:8px 10px; vertical-align:top; text-align:left; }}
.data-table th {{ background:var(--soft); font-weight:650; }}
.data-table td {{ color:#243043; }}
.empty-state {{ padding:24px; color:var(--muted); background:var(--soft); border-radius:8px; }}
.note {{ margin-top:8px; color:var(--muted); font-size:13px; }}
@media (max-width:900px) {{ header,main {{ padding-left:16px; padding-right:16px; }} .two-col {{ grid-template-columns:1fr; }} h1 {{ font-size:24px; }} nav a {{ padding:7px 9px; }} }}
</style></head><body>
<header><h1>Mapa semántico de tesis CIDE</h1><p>Mapa conjunto de licenciaturas, maestrías y doctorados con embeddings multilingües {html.escape(MODEL_NAME)}, UMAP, clustering comparativo y consenso ensemble. Generado el {html.escape(datetime.now().strftime('%Y-%m-%d %H:%M'))}.</p><nav><a href="#tiempo">Tiempo</a><a href="#mapa">Mapa</a><a href="#programas">Programas</a><a href="#topic-models">Topic models</a><a href="#consenso">Consenso</a><a href="#taxonomia">Taxonomía</a><a href="#clusters">Clusters</a><a href="#asesores">Profesorado</a><a href="#metodo">Método</a></nav></header>
<main><section class="metrics">{metric_html}</section><section class="grid">
<article class="panel" id="tiempo"><h2>Máquina temporal del mapa semántico</h2><p class="callout">El periodo más reciente tiene {latest_total} tesis en {latest_year}. El tema con mayor crecimiento relativo frente a 2010s es {html.escape(str(fastest_growth['cluster_theme']))} ({fastest_growth['cambio_2020s_vs_2010s_pp']:+.1f} pp).</p>{time_machine_html}</article>
<section class="two-col"><article class="panel"><h2>Mapa por periodos</h2>{periods_html}</article><article class="panel"><h2>Carrera anual de temas</h2>{race_html}</article></section>
<article class="panel"><h2>Composición temática anual</h2>{stream_html}</article>
<article class="panel"><h2>Ciclo de vida de los temas</h2>{lifecycle_table}</article>
<article class="panel" id="mapa"><h2>Mapa semántico acumulado</h2>{map_html}</article>
<section class="two-col"><article class="panel"><h2>Mapa semántico 3D exploratorio</h2><p class="note">La proyección 3D es para exploración interactiva; el clustering principal sigue calculándose sobre embeddings originales.</p>{map_3d_html}</article><article class="panel"><h2>UMAP 2D vs 3D</h2>{umap_quality_html}{umap_quality_table}</article></section>
<article class="panel" id="programas"><h2>El mapa por nivel académico</h2>{map_level_html}</article>
<section class="two-col"><article class="panel"><h2>Composición de temas por nivel</h2>{cluster_level_html}</article><article class="panel"><h2>Evolución de tesis por nivel</h2>{level_time_html}</article></section>
<article class="panel"><h2>Perfil temático de los programas</h2><p class="note">Cada fila suma 100%; los colores muestran qué temas concentran la producción de cada programa.</p>{program_cluster_html}</article>
<section class="two-col"><article class="panel"><h2>Similitud entre programas</h2><p class="callout">La pareja más cercana con al menos 10 tesis por programa es {html.escape(str(closest_program_pair['programa_a']))} y {html.escape(str(closest_program_pair['programa_b']))} (coseno {closest_program_pair['similitud_coseno']:.3f}).</p>{program_similarity_html}{program_similarity_table}</article><article class="panel"><h2>Clusters puente entre programas</h2>{interdisciplinarity_html}{interdisciplinarity_table}</article></section>
<article class="panel"><h2>Resumen temático por programa</h2>{program_summary_table}</article>
<section class="two-col" id="topic-models"><article class="panel"><h2>KMeans sobre UMAP 2D</h2>{variant_map_2d_html}</article><article class="panel"><h2>KMeans sobre UMAP 3D</h2>{variant_map_3d_html}</article></section>
<article class="panel"><h2>BERTopic + HDBSCAN</h2><p class="note">Modelo experimental inspirado en BERTopic: embeddings, UMAP 10D, HDBSCAN y c-TF-IDF. Los outliers aparecen como tema gris.</p>{bertopic_map_html}</article>
<article class="panel" id="consenso"><h2>Consenso entre modelos</h2><p class="note">Agrupa tesis que distintos métodos tienden a colocar juntas; es la vista recomendada para subtemas cuando importan estabilidad y bajo traslape.</p>{consensus_map_html}</article>
<section class="two-col"><article class="panel"><h2>Traslape de keywords por variante</h2><p class="note">Menor Jaccard indica menos keywords repetidas entre clusters; mayor c_v indica keywords más coherentes dentro de cada tema.</p>{variant_overlap_html}{variant_metrics_table}</article><article class="panel"><h2>Coherencia vs traslape</h2>{topic_quality_html}</article></section>
<section class="two-col"><article class="panel"><h2>Frontera Pareto de modelos</h2>{frontier_html}{frontier_table}</article><article class="panel"><h2>Estabilidad de variantes</h2>{stability_html}{stability_table}</article></section>
<section class="two-col" id="taxonomia"><article class="panel"><h2>Taxonomía macrotema → subtema</h2>{taxonomy_html}{taxonomy_table}</article><article class="panel"><h2>Membresía suave y tesis ambiguas</h2><p class="note">Un margen bajo indica tesis con mezcla temática; {ambiguous_share:.1%} cae en el cuartil más ambiguo.</p>{membership_html}{membership_table}</article></section>
<article class="panel"><h2>Pares de clusters con más keywords compartidas</h2>{keyword_overlap_table}</article>
<section class="two-col"><article class="panel"><h2>Diagnóstico de clusters</h2>{diag_html}<p class="note">Mejor k por silhouette: {int(best_diag['k'])}; k usado: {selected_k} para mayor granularidad interpretativa.</p></article><article class="panel" id="metodo"><h2>Benchmark de modelos</h2>{benchmark_table}</article></section>
<article class="panel" id="clusters"><h2>Resumen por cluster</h2>{cluster_table}</article>
<section class="two-col"><article class="panel"><h2>Heatmap año × tema</h2>{year_html}</article><article class="panel"><h2>Distribución por idioma y tema</h2>{language_html}</article></section>
<section class="two-col" id="asesores"><article class="panel"><h2>Profesorado: volumen, temas y programas</h2>{advisor_html}</article><article class="panel"><h2>Profesorado con más tesis</h2>{advisor_table}</article></section>
<article class="panel"><h2>Red de tesis por asesor y tema</h2>{network_html}</article>
</section></main></body></html>'''

DASHBOARD_PATH.write_text(html_doc, encoding='utf-8')
print(f'Dashboard HTML exportado: {DASHBOARD_PATH.resolve()}')
